In [1]:
"""Cricsheet exploration - turning ball-by-ball JSON into warehouse rows.

The ODI archive is downloaded by etl/01_download_cricsheet.py. This notebook
reads from it in place, without unpacking 579 MB onto disk.
"""

import json
import zipfile
from collections import Counter
from pathlib import Path


def find_project_root(start: Path) -> Path:
    """Walk upwards until we find the folder containing .git."""
    for candidate in [start, *start.parents]:
        if (candidate / ".git").exists():
            return candidate
    raise RuntimeError("Could not locate the project root")


PROJECT_ROOT = find_project_root(Path.cwd())
ODI_ZIP = PROJECT_ROOT / "data" / "raw" / "odis_json.zip"

print("project root:", PROJECT_ROOT)
print("archive     :", ODI_ZIP.name, f"({ODI_ZIP.stat().st_size / 1e6:.1f} MB)")

project root: d:\Tanay Nagpal\Labmentrix Internship\cricbuzz_livestats
archive     : odis_json.zip (21.5 MB)


In [2]:
with zipfile.ZipFile(ODI_ZIP) as archive:
    match_files = sorted(n for n in archive.namelist() if n.endswith(".json"))
    print(f"{len(match_files)} match files in the archive")

    sample_name = match_files[0]
    with archive.open(sample_name) as handle:
        match = json.load(handle)

print("reading:", sample_name)
print()
print("top-level keys:", list(match.keys()))
print()
print("meta:", match["meta"])
print()
print(f"info has {len(match['info'])} keys:")
for key in match["info"]:
    print("   ", key)
print()
print(f"innings: {len(match['innings'])}")

3182 match files in the archive
reading: 1000887.json

top-level keys: ['meta', 'info', 'innings']

meta: {'data_version': '1.2.0', 'created': '2017-01-14', 'revision': 2}

info has 18 keys:
    balls_per_over
    city
    dates
    event
    gender
    match_type
    match_type_number
    officials
    outcome
    overs
    player_of_match
    players
    registry
    season
    team_type
    teams
    toss
    venue

innings: 2


In [3]:
info = match["info"]

for key in ("match_type", "match_type_number", "team_type", "gender",
            "season", "dates", "venue", "city", "balls_per_over", "overs",
            "teams", "toss", "outcome", "player_of_match", "event"):
    print(f"{key:<18} {info.get(key)}")

match_type         ODI
match_type_number  3817
team_type          international
gender             male
season             2016/17
dates              ['2017-01-13']
venue              Brisbane Cricket Ground, Woolloongabba
city               Brisbane
balls_per_over     6
overs              50
teams              ['Australia', 'Pakistan']
toss               {'decision': 'bat', 'winner': 'Australia'}
outcome            {'by': {'runs': 92}, 'winner': 'Australia'}
player_of_match    ['MS Wade']
event              {'match_number': 1, 'name': 'Pakistan in Australia ODI Series'}


In [4]:
outcome_shapes = Counter()
by_keys = Counter()
result_values = Counter()

with zipfile.ZipFile(ODI_ZIP) as archive:
    for name in match_files:
        with archive.open(name) as handle:
            data = json.load(handle)

        outcome = data["info"].get("outcome", {})
        outcome_shapes[tuple(sorted(outcome.keys()))] += 1

        if "by" in outcome:
            by_keys[tuple(sorted(outcome["by"].keys()))] += 1
        if "result" in outcome:
            result_values[outcome["result"]] += 1

print("outcome key combinations:")
for keys, count in outcome_shapes.most_common():
    print(f"   {count:>5}  {keys}")

print("\n'by' key combinations:")
for keys, count in by_keys.most_common():
    print(f"   {count:>5}  {keys}")

print("\n'result' values:")
for value, count in result_values.most_common():
    print(f"   {count:>5}  {value}")

outcome key combinations:
    2767  ('by', 'winner')
     260  ('by', 'method', 'winner')
     140  ('result',)
       9  ('eliminator', 'result')
       5  ('method', 'result')
       1  ('eliminator', 'method', 'result')

'by' key combinations:
    1549  ('wickets',)
    1478  ('runs',)

'result' values:
     121  no result
      34  tie


In [5]:
seen = {}

with zipfile.ZipFile(ODI_ZIP) as archive:
    for name in match_files:
        with archive.open(name) as handle:
            data = json.load(handle)

        info = data["info"]
        shape = tuple(sorted(info.get("outcome", {}).keys()))

        if shape not in seen:
            seen[shape] = (name, info["teams"], info["dates"][0],
                           info["outcome"])

        if len(seen) == 6:
            break

for shape, (name, teams, date, outcome) in seen.items():
    print(f"{shape}")
    print(f"   {name}  {date}  {teams[0]} v {teams[1]}")
    print(f"   {outcome}")
    print()

('by', 'winner')
   1000887.json  2017-01-13  Australia v Pakistan
   {'by': {'runs': 92}, 'winner': 'Australia'}

('result',)
   1004283.json  2016-09-08  Scotland v Hong Kong
   {'result': 'no result'}

('by', 'method', 'winner')
   1022353.json  2017-06-04  India v Pakistan
   {'by': {'runs': 124}, 'method': 'D/L', 'winner': 'India'}

('eliminator', 'result')
   1144530.json  2019-07-14  England v New Zealand
   {'result': 'tie', 'eliminator': 'England'}

('method', 'result')
   1415981.json  2024-01-21  Zimbabwe v Ireland
   {'result': 'tie', 'method': 'D/L'}

('eliminator', 'method', 'result')
   1474411.json  2025-03-09  Canada v Namibia
   {'result': 'tie', 'eliminator': 'Canada', 'method': 'D/L'}



In [6]:
def parse_outcome(outcome: dict) -> dict:
    """Flatten Cricsheet's outcome object into fact_match columns.

    Six shapes appear across the ODI archive:
        by + winner                   normal win           2767
        by + method + winner          win on D/L            260
        result only                   no winner             140
        eliminator + result           super over win          9
        method + result               no winner, D/L          5
        eliminator + method + result  super over on D/L       1

    An 'eliminator' match DOES have a winner - the tie was broken by a
    super over - but carries no 'winner' key. The 2019 World Cup Final
    is one of these.
    """
    winner = outcome.get("winner")
    eliminator = outcome.get("eliminator")
    method = outcome.get("method")
    result = outcome.get("result")
    by = outcome.get("by") or {}

    if winner:
        # 'by' always holds exactly one key: runs or wickets.
        victory_type, margin = next(iter(by.items()), (None, None))
        return {"winner": winner, "victory_type": victory_type,
                "victory_margin": margin, "victory_method": method}

    if eliminator:
        return {"winner": eliminator, "victory_type": "super over",
                "victory_margin": None, "victory_method": method}

    return {"winner": None, "victory_type": result,
            "victory_margin": None, "victory_method": method}


# Verify against every match in the archive
types = Counter()
no_winner = 0

with zipfile.ZipFile(ODI_ZIP) as archive:
    for name in match_files:
        with archive.open(name) as handle:
            data = json.load(handle)

        parsed = parse_outcome(data["info"].get("outcome", {}))
        types[parsed["victory_type"]] += 1
        if parsed["winner"] is None:
            no_winner += 1

print("victory_type distribution:")
for value, count in types.most_common():
    print(f"   {count:>5}  {value}")

print(f"\nmatches with no winner: {no_winner}")
print(f"total: {sum(types.values())}")

victory_type distribution:
    1549  wickets
    1478  runs
     121  no result
      24  tie
      10  super over

matches with no winner: 145
total: 3182


In [7]:
genders = Counter()
team_types = Counter()
years = Counter()

with zipfile.ZipFile(ODI_ZIP) as archive:
    for name in match_files:
        with archive.open(name) as handle:
            info = json.load(handle)["info"]

        genders[info.get("gender")] += 1
        team_types[info.get("team_type")] += 1
        years[info["dates"][0][:4]] += 1

print("gender:")
for key, count in genders.most_common():
    print(f"   {count:>5}  {key}")

print("\nteam_type:")
for key, count in team_types.most_common():
    print(f"   {count:>5}  {key}")

print("\nmatches per year:")
for year in sorted(years):
    print(f"   {year}   {years[year]:>4}  {'#' * (years[year] // 4)}")

print(f"\ntotal        : {sum(years.values())}")
print(f"2006 onwards : {sum(v for k, v in years.items() if k >= '2006')}")

gender:
    2571  male
     611  female

team_type:
    3182  international

matches per year:
   2002      3  
   2003    119  #############################
   2004     79  ###################
   2005     67  ################
   2006    119  #############################
   2007    154  ######################################
   2008    104  ##########################
   2009    119  #############################
   2010    105  ##########################
   2011    140  ###################################
   2012     91  ######################
   2013    148  #####################################
   2014    118  #############################
   2015    138  ##################################
   2016    112  ############################
   2017    169  ##########################################
   2018    136  ##################################
   2019    167  #########################################
   2020     50  ############
   2021    110  ###########################
   2022    2

In [8]:
innings = match["innings"]
print(f"{len(innings)} innings\n")

first_innings = innings[0]
print("innings[0] keys:", list(first_innings.keys()))
print("team             :", first_innings["team"])
print("overs in innings :", len(first_innings["overs"]))
print()

first_over = first_innings["overs"][0]
print("over keys    :", list(first_over.keys()))
print("over number  :", first_over["over"])
print("deliveries   :", len(first_over["deliveries"]))
print()

for i, delivery in enumerate(first_over["deliveries"], start=1):
    print(f"ball {i}:")
    for key, value in delivery.items():
        print(f"      {key:<12} {value}")
    print()

2 innings

innings[0] keys: ['team', 'overs', 'powerplays']
team             : Australia
overs in innings : 50

over keys    : ['over', 'deliveries']
over number  : 0
deliveries   : 7

ball 1:
      actual_delivery 0.1
      batter       DA Warner
      bowler       Mohammad Amir
      non_striker  TM Head
      runs         {'batter': 0, 'extras': 0, 'total': 0}

ball 2:
      actual_delivery 0.2
      batter       DA Warner
      bowler       Mohammad Amir
      non_striker  TM Head
      runs         {'batter': 0, 'extras': 0, 'total': 0}

ball 3:
      actual_delivery 0.3
      batter       DA Warner
      bowler       Mohammad Amir
      non_striker  TM Head
      runs         {'batter': 0, 'extras': 0, 'total': 0}

ball 4:
      actual_delivery 0.4
      batter       DA Warner
      bowler       Mohammad Amir
      non_striker  TM Head
      runs         {'batter': 0, 'extras': 0, 'total': 0}

ball 5:
      actual_delivery 0.5
      batter       DA Warner
      bowler       Moham

In [9]:
extras_kinds = Counter()
wicket_kinds = Counter()
delivery_keys = Counter()
total_deliveries = 0

with zipfile.ZipFile(ODI_ZIP) as archive:
    for name in match_files[:300]:          # 300 matches is plenty for this
        with archive.open(name) as handle:
            data = json.load(handle)

        for innings in data.get("innings", []):
            for over in innings.get("overs", []):
                for delivery in over["deliveries"]:
                    total_deliveries += 1
                    delivery_keys[tuple(sorted(delivery.keys()))] += 1

                    for kind in delivery.get("extras", {}):
                        extras_kinds[kind] += 1

                    for wicket in delivery.get("wickets", []):
                        wicket_kinds[wicket["kind"]] += 1

print(f"deliveries scanned: {total_deliveries:,}\n")

print("delivery key combinations:")
for keys, count in delivery_keys.most_common():
    print(f"   {count:>7,}  {keys}")

print("\nextras kinds:")
for kind, count in extras_kinds.most_common():
    print(f"   {count:>7,}  {kind}")

print("\nwicket kinds:")
for kind, count in wicket_kinds.most_common():
    print(f"   {count:>7,}  {kind}")

deliveries scanned: 158,869

delivery key combinations:
   148,957  ('actual_delivery', 'batter', 'bowler', 'non_striker', 'runs')
     5,411  ('actual_delivery', 'batter', 'bowler', 'extras', 'non_striker', 'runs')
     4,114  ('actual_delivery', 'batter', 'bowler', 'non_striker', 'runs', 'wickets')
       178  ('actual_delivery', 'batter', 'bowler', 'non_striker', 'review', 'runs')
       143  ('actual_delivery', 'batter', 'bowler', 'non_striker', 'review', 'runs', 'wickets')
        31  ('actual_delivery', 'batter', 'bowler', 'extras', 'non_striker', 'review', 'runs')
        18  ('actual_delivery', 'batter', 'bowler', 'extras', 'non_striker', 'runs', 'wickets')
        16  ('actual_delivery', 'batter', 'bowler', 'non_striker', 'replacements', 'runs')
         1  ('actual_delivery', 'batter', 'bowler', 'non_striker', 'replacements', 'runs', 'wickets')

extras kinds:
     3,570  wides
     1,327  legbyes
       326  noballs
       237  byes
         2  penalty

wicket kinds:
     2,3

In [10]:
# Dismissals credited to the bowler. Run outs are not - the bowler had
# nothing to do with them. Retirements are not dismissals at all.
BOWLER_CREDITED = {"bowled", "caught", "lbw", "stumped",
                   "caught and bowled", "hit wicket"}

NOT_DISMISSALS = {"retired hurt", "retired not out"}


def parse_innings(innings: dict, innings_no: int) -> tuple[list, list]:
    """Aggregate one innings of deliveries into batting and bowling rows.

    Returns (batting_rows, bowling_rows) - one row per player per innings,
    which is the grain of fact_batting and fact_bowling.
    """
    batting_team = innings["team"]

    batting: dict[str, dict] = {}
    bowling: dict[str, dict] = {}
    order: list[str] = []

    def batter_row(name: str) -> dict:
        """Get or create a batter's row. Position = order of first appearance."""
        if name not in batting:
            order.append(name)
            batting[name] = {
                "player": name,
                "team": batting_team,
                "innings_no": innings_no,
                "batting_position": len(order),
                "runs_scored": 0,
                "balls_faced": 0,
                "fours": 0,
                "sixes": 0,
                "dismissal_kind": None,
                "dismissed_by": None,
                "fielder": None,
                "is_not_out": True,
            }
        return batting[name]

    def bowler_row(name: str) -> dict:
        if name not in bowling:
            bowling[name] = {
                "player": name,
                "innings_no": innings_no,
                "balls_bowled": 0,
                "runs_conceded": 0,
                "wickets": 0,
                "maidens": 0,
                "dots": 0,
            }
        return bowling[name]

    for over in innings.get("overs", []):
        over_runs_off_bat = 0
        over_illegal_balls = 0
        over_bowlers = set()

        for delivery in over["deliveries"]:
            runs = delivery["runs"]
            extras = delivery.get("extras", {})

            is_wide = "wides" in extras
            is_noball = "noballs" in extras

            striker = batter_row(delivery["batter"])
            batter_row(delivery["non_striker"])   # registers batting position
            bowler = bowler_row(delivery["bowler"])
            over_bowlers.add(delivery["bowler"])

            # ---- batting ------------------------------------------------
            striker["runs_scored"] += runs["batter"]

            if not is_wide:                       # a wide is not a ball faced
                striker["balls_faced"] += 1

            # non_boundary marks 4s and 6s that were run, not hit to the rope
            if not runs.get("non_boundary"):
                if runs["batter"] == 4:
                    striker["fours"] += 1
                elif runs["batter"] == 6:
                    striker["sixes"] += 1

            # ---- bowling ------------------------------------------------
            if is_wide or is_noball:
                over_illegal_balls += 1
            else:
                bowler["balls_bowled"] += 1
                if runs["total"] == 0:
                    bowler["dots"] += 1

            # Byes, leg-byes and penalties are NOT charged to the bowler.
            bowler["runs_conceded"] += (runs["batter"]
                                        + extras.get("wides", 0)
                                        + extras.get("noballs", 0))

            over_runs_off_bat += runs["batter"]

            # ---- wickets ------------------------------------------------
            for wicket in delivery.get("wickets", []):
                kind = wicket["kind"]

                if kind in NOT_DISMISSALS:        # retired hurt: still not out
                    continue

                dismissed = batter_row(wicket["player_out"])
                dismissed["is_not_out"] = False
                dismissed["dismissal_kind"] = kind

                fielders = wicket.get("fielders") or []
                if fielders:
                    first = fielders[0]
                    dismissed["fielder"] = (first.get("name")
                                            if isinstance(first, dict) else first)

                if kind in BOWLER_CREDITED:
                    dismissed["dismissed_by"] = delivery["bowler"]
                    bowler["wickets"] += 1

        # A maiden: no runs off the bat, and no wides or no-balls.
        # Byes and leg-byes don't spoil a maiden - they aren't the bowler's fault.
        if (over_runs_off_bat == 0 and over_illegal_balls == 0
                and len(over_bowlers) == 1):
            bowling[next(iter(over_bowlers))]["maidens"] += 1

    return list(batting.values()), list(bowling.values())


print("parse_innings ready")

parse_innings ready


In [11]:
batting_rows, bowling_rows = parse_innings(match["innings"][0], 1)

print(f"{match['info']['teams'][0]} v {match['info']['teams'][1]}, "
      f"{match['info']['dates'][0]}")
print(f"innings 1 - {match['innings'][0]['team']}\n")

print(f"{'#':<3}{'batter':<22}{'R':>5}{'B':>5}{'4s':>4}{'6s':>4}   dismissal")
print("-" * 78)
for row in sorted(batting_rows, key=lambda r: r["batting_position"]):
    out = "not out" if row["is_not_out"] else row["dismissal_kind"]
    print(f"{row['batting_position']:<3}{row['player']:<22}"
          f"{row['runs_scored']:>5}{row['balls_faced']:>5}"
          f"{row['fours']:>4}{row['sixes']:>4}   {out}")

print(f"\n{'bowler':<22}{'O':>7}{'M':>4}{'R':>5}{'W':>4}{'econ':>7}")
print("-" * 50)
for row in bowling_rows:
    overs = f"{row['balls_bowled'] // 6}.{row['balls_bowled'] % 6}"
    econ = row["runs_conceded"] * 6 / row["balls_bowled"] if row["balls_bowled"] else 0
    print(f"{row['player']:<22}{overs:>7}{row['maidens']:>4}"
          f"{row['runs_conceded']:>5}{row['wickets']:>4}{econ:>7.2f}")

print(f"\nruns off the bat : {sum(r['runs_scored'] for r in batting_rows)}")
print(f"wickets fallen   : {sum(1 for r in batting_rows if not r['is_not_out'])}")
print(f"balls bowled     : {sum(r['balls_bowled'] for r in bowling_rows)}")

Australia v Pakistan, 2017-01-13
innings 1 - Australia

#  batter                    R    B  4s  6s   dismissal
------------------------------------------------------------------------------
1  DA Warner                 7   18   1   0   bowled
2  TM Head                  39   39   5   0   caught
3  SPD Smith                 0    1   0   0   caught
4  CA Lynn                  16   12   0   1   caught
5  MR Marsh                  4   17   0   0   caught
6  GJ Maxwell               60   56   7   0   caught
7  MS Wade                 100  100   7   2   not out
8  JP Faulkner               5   12   0   0   caught
9  PJ Cummins               15   32   0   0   run out
10 MA Starc                 10    8   0   1   bowled
11 B Stanlake                1    6   0   0   not out

bowler                      O   M    R   W   econ
--------------------------------------------------
Mohammad Amir            10.0   0   54   2   5.40
Mohammad Hafeez           7.0   0   23   0   3.29
Hasan Ali            

In [16]:
BOWLER_CREDITED = {"bowled", "caught", "lbw", "stumped",
                   "caught and bowled", "hit wicket"}

NOT_DISMISSALS = {"retired hurt", "retired not out"}


def parse_innings(innings: dict, innings_no: int) -> tuple[list, list]:
    """Aggregate one innings of deliveries into batting and bowling rows.

    Returns (batting_rows, bowling_rows) - one row per player per innings,
    which is the grain of fact_batting and fact_bowling.
    """
    # A super over is a tiebreak, not an innings. Rows from it would give
    # players extra "innings" and corrupt every batting average.
    if innings.get("super_over"):
        return [], []

    batting_team = innings["team"]

    batting: dict[str, dict] = {}
    bowling: dict[str, dict] = {}
    order: list[str] = []

    def batter_row(name: str) -> dict:
        """Get or create a batter's row. Position = order of first appearance."""
        if name not in batting:
            order.append(name)
            batting[name] = {
                "player": name,
                "team": batting_team,
                "innings_no": innings_no,
                "batting_position": len(order),
                "runs_scored": 0,
                "balls_faced": 0,
                "fours": 0,
                "sixes": 0,
                "dismissal_kind": None,
                "dismissed_by": None,
                "fielder": None,
                "is_not_out": True,
            }
        return batting[name]

    def bowler_row(name: str) -> dict:
        if name not in bowling:
            bowling[name] = {
                "player": name,
                "innings_no": innings_no,
                "balls_bowled": 0,
                "runs_conceded": 0,
                "wickets": 0,
                "maidens": 0,
                "dots": 0,
            }
        return bowling[name]

    for over in innings.get("overs", []):
        over_runs_off_bat = 0
        over_illegal_balls = 0
        over_bowlers = set()

        for delivery in over["deliveries"]:
            runs = delivery["runs"]
            extras = delivery.get("extras", {})

            is_wide = "wides" in extras
            is_noball = "noballs" in extras

            striker = batter_row(delivery["batter"])
            batter_row(delivery["non_striker"])   # registers batting position
            bowler = bowler_row(delivery["bowler"])
            over_bowlers.add(delivery["bowler"])

            # ---- batting ------------------------------------------------
            striker["runs_scored"] += runs["batter"]

            if not is_wide:                       # a wide is not a ball faced
                striker["balls_faced"] += 1

            # non_boundary marks 4s and 6s that were run, not hit to the rope
            if not runs.get("non_boundary"):
                if runs["batter"] == 4:
                    striker["fours"] += 1
                elif runs["batter"] == 6:
                    striker["sixes"] += 1

            # ---- bowling ------------------------------------------------
            if is_wide or is_noball:
                over_illegal_balls += 1
            else:
                bowler["balls_bowled"] += 1
                if runs["total"] == 0:
                    bowler["dots"] += 1

            # Byes, leg-byes and penalties are NOT charged to the bowler.
            bowler["runs_conceded"] += (runs["batter"]
                                        + extras.get("wides", 0)
                                        + extras.get("noballs", 0))

            over_runs_off_bat += runs["batter"]

            # ---- wickets ------------------------------------------------
            for wicket in delivery.get("wickets", []):
                kind = wicket["kind"]

                if kind in NOT_DISMISSALS:        # retired hurt: still not out
                    continue

                dismissed = batter_row(wicket["player_out"])
                dismissed["is_not_out"] = False
                dismissed["dismissal_kind"] = kind

                fielders = wicket.get("fielders") or []
                if fielders:
                    first = fielders[0]
                    dismissed["fielder"] = (first.get("name")
                                            if isinstance(first, dict) else first)

                if kind in BOWLER_CREDITED:
                    dismissed["dismissed_by"] = delivery["bowler"]
                    bowler["wickets"] += 1

        # A maiden: no runs off the bat, and no wides or no-balls.
        # Byes and leg-byes don't spoil a maiden - not the bowler's fault.
        if (over_runs_off_bat == 0 and over_illegal_balls == 0
                and len(over_bowlers) == 1):
            bowling[next(iter(over_bowlers))]["maidens"] += 1

    return list(batting.values()), list(bowling.values())


print("parse_innings ready (super overs excluded)")

parse_innings ready (super overs excluded)


In [17]:
errors = []
anomalies = []
innings_counts = Counter()
innings_keys = Counter()
positions_seen = Counter()

total_batting = 0
total_bowling = 0

with zipfile.ZipFile(ODI_ZIP) as archive:
    for name in match_files:
        with archive.open(name) as handle:
            data = json.load(handle)

        all_innings = data.get("innings", [])
        innings_counts[len(all_innings)] += 1

        for number, one_innings in enumerate(all_innings, start=1):
            innings_keys[tuple(sorted(one_innings.keys()))] += 1

            try:
                batting_rows, bowling_rows = parse_innings(one_innings, number)
            except Exception as exc:
                errors.append((name, number, f"{type(exc).__name__}: {exc}"))
                continue

            total_batting += len(batting_rows)
            total_bowling += len(bowling_rows)

            for row in batting_rows:
                positions_seen[row["batting_position"]] += 1

                if row["balls_faced"] == 0 and row["runs_scored"] > 0:
                    anomalies.append((name, "runs without balls", row["player"]))
                if row["runs_scored"] < 0 or row["balls_faced"] < 0:
                    anomalies.append((name, "negative value", row["player"]))

print(f"batting rows : {total_batting:,}")
print(f"bowling rows : {total_bowling:,}")
print(f"\ncrashes   : {len(errors)}")
for item in errors[:5]:
    print("   ", item)
print(f"\nanomalies : {len(anomalies)}")
for item in anomalies[:5]:
    print("   ", item)
print("\ninnings per match:")
for count, matches in sorted(innings_counts.items()):
    print(f"   {matches:>5} matches have {count} innings")

batting rows : 56,052
bowling rows : 38,403

crashes   : 0

anomalies : 0

innings per match:
      81 matches have 1 innings
    3091 matches have 2 innings
      10 matches have 4 innings


In [18]:
import sys

# The notebook lives in notebooks/, so the project root isn't importable
# by default. Add it so we can reuse utils/ rather than duplicating code.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils.transformers import normalise_format


def parse_match_info(info: dict, match_id: int) -> dict:
    """Flatten Cricsheet's info block into one fact_match row.

    match_id comes from the filename - Cricsheet stores no id inside the file.
    """
    outcome = parse_outcome(info.get("outcome", {}))
    event = info.get("event") or {}
    toss = info.get("toss") or {}
    teams = info.get("teams") or [None, None]
    player_of_match = info.get("player_of_match") or []
    dates = info.get("dates") or [None]

    return {
        "match_id": match_id,
        "match_format": normalise_format(info.get("match_type")),
        "match_type_number": info.get("match_type_number"),
        "gender": info.get("gender"),
        "team_type": info.get("team_type"),
        "season": info.get("season"),

        # Tests span several days; dates[0] is the match date.
        "match_date": dates[0],
        "end_date": dates[-1],

        "venue": info.get("venue"),
        "city": info.get("city"),

        "team1": teams[0] if len(teams) > 0 else None,
        "team2": teams[1] if len(teams) > 1 else None,

        "toss_winner": toss.get("winner"),
        "toss_decision": toss.get("decision"),

        "winner": outcome["winner"],
        "victory_type": outcome["victory_type"],
        "victory_margin": outcome["victory_margin"],
        "victory_method": outcome["victory_method"],

        "player_of_match": player_of_match[0] if player_of_match else None,

        "series_name": event.get("name"),
        "event_match_number": event.get("match_number"),

        "balls_per_over": info.get("balls_per_over"),
        "scheduled_overs": info.get("overs"),
    }


print("parse_match_info ready")

parse_match_info ready


In [19]:
match_id = int(sample_name.removesuffix(".json"))
row = parse_match_info(match["info"], match_id)

for key, value in row.items():
    print(f"{key:<20} {value}")

match_id             1000887
match_format         ODI
match_type_number    3817
gender               male
team_type            international
season               2016/17
match_date           2017-01-13
end_date             2017-01-13
venue                Brisbane Cricket Ground, Woolloongabba
city                 Brisbane
team1                Australia
team2                Pakistan
toss_winner          Australia
toss_decision        bat
winner               Australia
victory_type         runs
victory_margin       92
victory_method       None
player_of_match      MS Wade
series_name          Pakistan in Australia ODI Series
event_match_number   1
balls_per_over       6
scheduled_overs      50


In [20]:
null_counts = Counter()
formats = Counter()
team_counts = Counter()
match_rows = []

with zipfile.ZipFile(ODI_ZIP) as archive:
    for name in match_files:
        with archive.open(name) as handle:
            info = json.load(handle)["info"]

        row = parse_match_info(info, int(name.removesuffix(".json")))
        match_rows.append(row)

        formats[row["match_format"]] += 1
        team_counts[len(info.get("teams") or [])] += 1

        for key, value in row.items():
            if value is None:
                null_counts[key] += 1

print(f"match rows: {len(match_rows):,}\n")

print("formats:")
for value, count in formats.most_common():
    print(f"   {count:>5}  {value}")

print("\nteams listed per match:")
for count, matches in sorted(team_counts.items()):
    print(f"   {matches:>5} matches list {count} teams")

print("\nfields containing NULLs:")
for key, count in null_counts.most_common():
    pct = count * 100 / len(match_rows)
    print(f"   {count:>5}  ({pct:5.1f}%)  {key}")

match rows: 3,182

formats:
    3182  ODI

teams listed per match:
    3182 matches list 2 teams

fields containing NULLs:
    2916  ( 91.6%)  victory_method
     306  (  9.6%)  city
     222  (  7.0%)  player_of_match
     214  (  6.7%)  event_match_number
     155  (  4.9%)  victory_margin
     145  (  4.6%)  winner
      14  (  0.4%)  series_name


In [21]:
import pandas as pd

register = pd.read_csv(PROJECT_ROOT / "data" / "raw" / "people.csv")
ours = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "people.parquet")

print("register :", register.shape)
print("columns  :", list(register.columns))
print()
print(register.head(3).to_string(index=False))

print(f"\nour players: {len(ours):,}")

merged = ours.merge(register, left_on="cricsheet_id",
                    right_on="identifier", how="left")

print(f"\nmatched to register : {merged['identifier'].notna().sum():,} "
      f"of {len(ours):,}")
print(f"have key_cricbuzz   : {merged['key_cricbuzz'].notna().sum():,}")
print(f"have key_cricinfo   : {merged['key_cricinfo'].notna().sum():,}")

print("\n--- the two SR Taylors ---")
print(register[register["unique_name"] == "SR Taylor"][
    ["identifier", "name", "unique_name", "key_cricbuzz"]
].to_string(index=False))

register : (18539, 22)
columns  : ['identifier', 'name', 'unique_name', 'key_bcci', 'key_bcci_2', 'key_bigbash', 'key_cricbuzz', 'key_cricheroes', 'key_crichq', 'key_cricinfo', 'key_cricinfo_2', 'key_cricinfo_3', 'key_cricingif', 'key_cricketarchive', 'key_cricketarchive_2', 'key_cricketworld', 'key_nvplay', 'key_nvplay_2', 'key_opta', 'key_opta_2', 'key_pulse', 'key_pulse_2']

identifier          name   unique_name key_bcci key_bcci_2  key_bigbash  key_cricbuzz  key_cricheroes  key_crichq  key_cricinfo  key_cricinfo_2  key_cricinfo_3  key_cricingif  key_cricketarchive  key_cricketarchive_2  key_cricketworld key_nvplay key_nvplay_2  key_opta  key_opta_2  key_pulse  key_pulse_2
  b4a23876 AAA Amsterdam AAA Amsterdam      NaN        NaN          NaN           NaN             NaN         NaN      772407.0             NaN             NaN            NaN                 NaN                   NaN               NaN        NaN          NaN       NaN         NaN        NaN          NaN
  482762a

In [22]:
print(register[register["identifier"].isin(["92cf79a8", "a9231c3f"])][
    ["identifier", "name", "unique_name", "key_cricinfo"]
].to_string(index=False))

identifier      name        unique_name  key_cricinfo
  92cf79a8 SR Taylor          SR Taylor      355359.0
  a9231c3f SR Taylor Steven Ryan Taylor      348133.0


In [23]:
PROCESSED = PROJECT_ROOT / "data" / "processed"

matches = pd.read_parquet(PROCESSED / "matches.parquet")
batting = pd.read_parquet(PROCESSED / "batting.parquet")
bowling = pd.read_parquet(PROCESSED / "bowling.parquet")

print(f"matches {len(matches):,}   batting {len(batting):,}   "
      f"bowling {len(bowling):,}")

match_gender = matches.set_index("match_id")["gender"]

# Every appearance, batting or bowling. A specialist bowler who was never
# dismissed has no batting row, so neither table alone is enough.
appearances = pd.concat([
    batting[["player_id", "match_id", "team"]],
    bowling[["player_id", "match_id", "team"]],
]).dropna(subset=["player_id"]).drop_duplicates()

appearances["gender"] = appearances["match_id"].map(match_gender)

# --- how much did they bat? ---
bat_profile = batting.dropna(subset=["player_id"]).groupby("player_id").agg(
    innings_batted=("match_id", "nunique"),
    avg_position=("batting_position", "mean"),
    total_runs=("runs_scored", "sum"),
    balls_faced=("balls_faced", "sum"),
)

# --- how much did they bowl? ---
bowl_profile = bowling.dropna(subset=["player_id"]).groupby("player_id").agg(
    innings_bowled=("match_id", "nunique"),
    balls_bowled=("balls_bowled", "sum"),
    wickets=("wickets", "sum"),
)

# --- stumpings identify a keeper: only a keeper can stump someone ---
stumpings = (batting[batting["dismissal_kind"] == "stumped"]
             .dropna(subset=["fielder_id"])
             .groupby("fielder_id").size()
             .rename("stumpings"))
stumpings.index.name = "player_id"

matches_played = (appearances.groupby("player_id")["match_id"].nunique()
                  .rename("matches_played"))

country = (appearances.groupby(["player_id", "team"]).size()
           .rename("n").reset_index()
           .sort_values("n", ascending=False)
           .drop_duplicates("player_id")
           .set_index("player_id")["team"].rename("country"))

gender = (appearances.groupby(["player_id", "gender"]).size()
          .rename("n").reset_index()
          .sort_values("n", ascending=False)
          .drop_duplicates("player_id")
          .set_index("player_id")["gender"])

profile = (pd.concat([matches_played, bat_profile, bowl_profile,
                      stumpings, country, gender], axis=1)
           .fillna({"innings_batted": 0, "innings_bowled": 0,
                    "total_runs": 0, "balls_faced": 0,
                    "balls_bowled": 0, "wickets": 0, "stumpings": 0}))

profile["bowl_share"] = (profile["innings_bowled"]
                         / profile["matches_played"]).fillna(0)

print(f"\nprofiled players: {len(profile):,}")
print()
print(profile.sort_values("matches_played", ascending=False)
      .head(10)
      .to_string())

matches 3,182   batting 56,052   bowling 38,403

profiled players: 2,682

           matches_played  innings_batted  avg_position  total_runs  balls_faced  innings_bowled  balls_bowled  wickets  stumpings     country gender  bowl_share
player_id                                                                                                                                                        
ba607b88            301.0           300.0      3.183333     14819.0      15784.0            50.0         662.0      5.0        0.0       India   male    0.166113
6b71e6cf            284.0           284.0      3.285211     11640.0      14315.0             0.0           0.0      0.0       75.0   Sri Lanka   male    0.000000
4a8a2e3b            281.0           279.0      5.469534     10274.0      11820.0             2.0          36.0      1.0      118.0       India   male    0.007117
740742ef            274.0           273.0      2.234432     11532.0      12478.0            40.0         610.0      

In [24]:
# Who's in the registry but never batted or bowled?
registry_ids = set(ours["cricsheet_id"])
profiled_ids = set(profile.index)
missing_ids = registry_ids - profiled_ids

print(f"in registry, never batted or bowled: {len(missing_ids):,}")

# Are they officials? Check one match's people vs its officials.
with zipfile.ZipFile(ODI_ZIP) as archive:
    with archive.open(sample_name) as handle:
        sample = json.load(handle)["info"]

officials = sample.get("officials", {})
official_names = {name for names in officials.values() for name in names}
squad_names = {n for team in sample.get("players", {}).values() for n in team}
registry_names = set((sample.get("registry") or {}).get("people", {}))

print(f"\nsample match {sample_name}")
print(f"   registry names : {len(registry_names)}")
print(f"   squad names    : {len(squad_names)}")
print(f"   official names : {len(official_names)}")
print(f"   officials in registry: "
      f"{len(official_names & registry_names)} of {len(official_names)}")
print(f"   registry minus squads minus officials: "
      f"{len(registry_names - squad_names - official_names)}")

print("\n   officials:", officials)

# Now name some of the missing players, via the register
missing_named = ours[ours["cricsheet_id"].isin(list(missing_ids)[:15])]
print("\nsample of the missing:")
print(missing_named[["cricsheet_id", "player_name"]].head(15).to_string(index=False))

in registry, never batted or bowled: 580

sample match 1000887.json
   registry names : 27
   squad names    : 22
   official names : 5
   officials in registry: 5 of 5
   registry minus squads minus officials: 0

   officials: {'match_referees': ['JJ Crowe'], 'reserve_umpires': ['SJ Nogajski'], 'tv_umpires': ['CB Gaffaney'], 'umpires': ['MD Martell', 'C Shamshuddin']}

sample of the missing:
cricsheet_id    player_name
    26c01625     Adele Louw
    3424e050         R Shah
    4a3132b1    TJ Matibiri
    4c47ebf2      GC Joshua
    5e39110e        I Chabi
    70fdd428     JD Shantry
    800187dc     SR Bernard
    8387d138  Nadeem Ghauri
    84424f4f A Nand Kishore
    85143097      I Shivram
    8f9e95fd          S Rao
    90d95b82      AY Harris
    93a6d2d0      PJ Vosloo
    e5c6f48b       RA Dykes
    f7e8ee52     BC Treloar


In [25]:
squads = pd.read_parquet(PROCESSED / "squads.parquet")

# dim_player's population is squad membership, not the registry.
player_ids = set(squads["player_id"].dropna())
print(f"players: {len(player_ids):,}")


def classify_role(row) -> str:
    """Infer a playing role from what the player actually did.

    Only a wicket-keeper can stump someone, so a stumping is proof rather
    than a hint. The rest is judgement: how often they bowled, and how high
    they batted.
    """
    if row["stumpings"] > 0:
        return "Wicket-keeper"

    if row["bowl_share"] >= 0.4:
        return "All-rounder" if row["avg_position"] <= 7 else "Bowler"

    return "Batsman"


roles = profile.loc[profile.index.isin(player_ids)].copy()
roles["playing_role"] = roles.apply(classify_role, axis=1)

print("\nrole distribution:")
counts = roles["playing_role"].value_counts()
for role, count in counts.items():
    print(f"   {count:>5}  ({count / len(roles):5.1%})  {role}")

print("\ntop 15 by matches played, with derived role:")
top = (roles.sort_values("matches_played", ascending=False).head(15)
       .merge(register.set_index("identifier")[["unique_name"]],
              left_index=True, right_index=True, how="left"))

print(top[["unique_name", "country", "matches_played", "avg_position",
           "total_runs", "wickets", "stumpings", "bowl_share",
           "playing_role"]].to_string())

players: 2,694

role distribution:
    1224  (45.6%)  Bowler
     792  (29.5%)  Batsman
     483  (18.0%)  All-rounder
     183  ( 6.8%)  Wicket-keeper

top 15 by matches played, with derived role:
                unique_name       country  matches_played  avg_position  total_runs  wickets  stumpings  bowl_share   playing_role
player_id                                                                                                                         
ba607b88            V Kohli         India           301.0      3.183333     14819.0      5.0        0.0    0.166113        Batsman
6b71e6cf      KC Sangakkara     Sri Lanka           284.0      3.285211     11640.0      0.0       75.0    0.000000  Wicket-keeper
4a8a2e3b           MS Dhoni         India           281.0      5.469534     10274.0      1.0      118.0    0.007117  Wicket-keeper
740742ef          RG Sharma         India           274.0      2.234432     11532.0      9.0        0.0    0.145985        Batsman
5bdcdb72        

In [26]:
keepers = roles[roles["stumpings"] > 0].copy()
keepers["stump_rate"] = keepers["stumpings"] / keepers["matches_played"]

print(f"players with at least one stumping: {len(keepers):,}\n")

print("stumping-rate distribution:")
for low, high in [(0, 0.02), (0.02, 0.05), (0.05, 0.10),
                  (0.10, 0.20), (0.20, 1.0)]:
    n = ((keepers["stump_rate"] >= low) & (keepers["stump_rate"] < high)).sum()
    print(f"   {low:.2f} - {high:.2f}   {n:>4}")

print("\nlowest 15 rates (likely part-time keepers):")
low = (keepers.sort_values("stump_rate").head(15)
       .merge(register.set_index("identifier")[["unique_name"]],
              left_index=True, right_index=True, how="left"))
print(low[["unique_name", "country", "matches_played", "stumpings",
           "stump_rate", "total_runs", "avg_position"]].to_string())

players with at least one stumping: 183

stumping-rate distribution:
   0.00 - 0.02      2
   0.02 - 0.05     17
   0.05 - 0.10     31
   0.10 - 0.20     48
   0.20 - 1.00     75

lowest 15 rates (likely part-time keepers):
               unique_name                   country  matches_played  stumpings  stump_rate  total_runs  avg_position
player_id                                                                                                            
b2664905       TT Beaumont                   England           112.0        1.0    0.008929      4400.0      1.696429
3241e3fd          N Pooran               West Indies            54.0        1.0    0.018519      1829.0      4.592593
c4487b84    AB de Villiers              South Africa           213.0        5.0    0.023474      9435.0      3.765258
52d1dbc8         BL Mooney                 Australia            84.0        2.0    0.023810      3155.0      3.809524
7b01ce83             L Lee              South Africa            76.0

In [27]:
# Catches taken, from the fielder credited on each dismissal.
catches = (batting[batting["dismissal_kind"] == "caught"]
           .dropna(subset=["fielder_id"])
           .groupby("fielder_id").size()
           .rename("catches"))
catches.index.name = "player_id"

fielding = roles.join(catches).fillna({"catches": 0})
fielding["dismissals"] = fielding["catches"] + fielding["stumpings"]
fielding["dismissal_rate"] = (fielding["dismissals"]
                              / fielding["matches_played"])

named = fielding.merge(register.set_index("identifier")[["unique_name"]],
                       left_index=True, right_index=True, how="left")

experienced = named[named["matches_played"] >= 30]
print(f"players with 30+ matches: {len(experienced):,}\n")

print("top 20 by dismissal rate:")
print(experienced.sort_values("dismissal_rate", ascending=False).head(20)[
    ["unique_name", "country", "matches_played", "catches", "stumpings",
     "dismissal_rate", "playing_role"]].to_string())

print("\nAB de Villiers and Rishabh Pant for comparison:")
print(named.loc[named.index.isin(["c4487b84", "919a3be2"])][
    ["unique_name", "matches_played", "catches", "stumpings",
     "dismissal_rate"]].to_string())

players with 30+ matches: 693

top 20 by dismissal rate:
             unique_name       country  matches_played  catches  stumpings  dismissal_rate   playing_role
player_id                                                                                                
2e929b99        GO Jones       England            34.0     59.0        4.0        1.852941  Wicket-keeper
2b6e6dec    AC Gilchrist     Australia           118.0    186.0       19.0        1.737288  Wicket-keeper
f3cb53a1      MV Boucher  South Africa           106.0    173.0        9.0        1.716981  Wicket-keeper
0fa5042b        L Ronchi   New Zealand            68.0    104.0       12.0        1.705882  Wicket-keeper
3979b5e0        L Tucker       Ireland            43.0     70.0        3.0        1.697674  Wicket-keeper
d1523761        D Ramdin   West Indies           103.0    164.0        7.0        1.660194  Wicket-keeper
53a5767d        T Chetty  South Africa            56.0     67.0       25.0        1.642857  Wic

In [28]:
band = experienced[(experienced["dismissal_rate"] >= 0.70)
                   & (experienced["dismissal_rate"] < 1.40)]

print(f"players with 30+ matches in the 0.70 - 1.40 band: {len(band)}\n")

print(band.sort_values("dismissal_rate", ascending=False)[
    ["unique_name", "country", "matches_played", "catches", "stumpings",
     "dismissal_rate", "avg_position", "bowl_share"]].to_string())

players with 30+ matches in the 0.70 - 1.40 band: 40

               unique_name               country  matches_played  catches  stumpings  dismissal_rate  avg_position  bowl_share
player_id                                                                                                                     
8b261d84          MH Cross              Scotland            91.0    116.0       11.0        1.395604      3.296703    0.000000
afa7e784           MS Wade             Australia            82.0    104.0        9.0        1.378049      5.670732    0.000000
8716233e          ZE Green               Namibia            47.0     57.0        6.0        1.340426      5.297872    0.000000
1200550f          MJ Prior               England            58.0     69.0        7.0        1.310345      3.603448    0.000000
2f26ac1a   Mohammad Rizwan              Pakistan            90.0    107.0        7.0        1.266667      4.644444    0.000000
b05e45e4          AE Jones               England         

In [29]:
def classify_role(row) -> str:
    """Infer a playing role from what the player actually did.

    Order matters. Bowling is checked first because a keeper cannot bowl
    while keeping - so a high bowl_share means the gloves were incidental
    (GD Phillips keeps, but plays for NZ as a batsman who bowls off-spin).

    Keepers are identified by dismissal rate, not stumpings. Stumpings only
    happen off spin, so a stumping rate measures how much spin a team bowled
    rather than whether someone kept wicket - by that test Rishabh Pant,
    Nicholas Pooran and Jonny Bairstow all look like batsmen. Catches are
    dense where stumpings are sparse.

    Known limitation: AB de Villiers classifies as a keeper. He kept in many
    ODIs, so it is defensible, but he is primarily a batsman. Any threshold
    that excluded him would also exclude four unambiguous keepers.
    """
    if row["bowl_share"] >= 0.4:
        return "All-rounder" if row["avg_position"] <= 7 else "Bowler"

    if row["dismissal_rate"] >= 0.70:
        return "Wicket-keeper"

    return "Batsman"


fielding["playing_role"] = fielding.apply(classify_role, axis=1)

print("role distribution:")
counts = fielding["playing_role"].value_counts()
for role, count in counts.items():
    print(f"   {count:>5}  ({count / len(fielding):5.1%})  {role}")

check = (fielding.sort_values("matches_played", ascending=False).head(15)
         .merge(register.set_index("identifier")[["unique_name"]],
                left_index=True, right_index=True, how="left"))

print("\ntop 15 with the final rule:")
print(check[["unique_name", "country", "matches_played", "dismissal_rate",
             "bowl_share", "playing_role"]].to_string())

role distribution:
    1224  (45.6%)  Bowler
     745  (27.8%)  Batsman
     484  (18.0%)  All-rounder
     229  ( 8.5%)  Wicket-keeper

top 15 with the final rule:
                unique_name       country  matches_played  dismissal_rate  bowl_share   playing_role
player_id                                                                                           
ba607b88            V Kohli         India           301.0        0.548173    0.166113        Batsman
6b71e6cf      KC Sangakkara     Sri Lanka           284.0        1.404930    0.000000  Wicket-keeper
4a8a2e3b           MS Dhoni         India           281.0        1.516014    0.007117  Wicket-keeper
740742ef          RG Sharma         India           274.0        0.386861    0.145985        Batsman
5bdcdb72         TM Dilshan     Sri Lanka           267.0        0.344569    0.674157    All-rounder
d18f9182   DPMD Jayawardene     Sri Lanka           264.0        0.530303    0.018939        Batsman
a94e08ea    Mushfiqur Rahim

In [30]:
venues = (matches.groupby("venue")
          .agg(matches_played=("match_id", "count"),
               cities=("city", lambda s: s.dropna().nunique()),
               city=("city", lambda s: s.dropna().iloc[0]
                     if s.notna().any() else None))
          .sort_values("matches_played", ascending=False))

print(f"distinct venues: {len(venues)}")
print(f"venues with no city at all: {venues['city'].isna().sum()}")
print(f"venues with >1 city spelling: {(venues['cities'] > 1).sum()}")

print("\ntop 20 venues:")
print(venues.head(20).to_string())

print("\nvenues with no city:")
print(venues[venues["city"].isna()].head(20).to_string())

distinct venues: 382
venues with no city at all: 9
venues with >1 city spelling: 6

top 20 venues:
                                                             matches_played  cities        city
venue                                                                                          
Harare Sports Club                                                      127       1      Harare
Shere Bangla National Stadium                                            85       2      Mirpur
Dubai International Cricket Stadium                                      59       1       Dubai
Rangiri Dambulla International Stadium                                   51       1    Dambulla
R Premadasa Stadium                                                      49       1     Colombo
Sydney Cricket Ground                                                    49       1      Sydney
R Premadasa Stadium, Colombo                                             45       1     Colombo
Kennington Oval                      

In [31]:
# Fill missing cities from the venue's other matches
venue_city = (matches.dropna(subset=["city"])
              .groupby("venue")["city"]
              .agg(lambda s: s.value_counts().idxmax()))

# Derive a country: the team that appears at this venue most often.
# Home sides play far more at their own grounds than visitors do.
venue_teams = pd.concat([
    matches[["venue", "team1"]].rename(columns={"team1": "team"}),
    matches[["venue", "team2"]].rename(columns={"team2": "team"}),
])
venue_country = (venue_teams.groupby(["venue", "team"]).size()
                 .rename("n").reset_index()
                 .sort_values("n", ascending=False)
                 .drop_duplicates("venue")
                 .set_index("venue")["team"].rename("likely_country"))

review = (matches.groupby("venue")
          .agg(matches_played=("match_id", "count"),
               first_match=("match_date", "min"),
               last_match=("match_date", "max"),
               distinct_cities=("city", lambda s: s.dropna().nunique()))
          .join(venue_city.rename("city"))
          .join(venue_country)
          .sort_values("matches_played", ascending=False))

# Flag venues whose name is contained in another venue's name -
# the "R Premadasa Stadium" / "R Premadasa Stadium, Colombo" pattern.
names = list(review.index)
review["name_overlap"] = [
    "|".join(o for o in names if o != n and (n in o or o in n)) or None
    for n in names
]

review["capacity"] = None      # for you to fill in
review["country"] = review["likely_country"]   # for you to correct

out = PROCESSED / "venues_review.csv"
review.to_csv(out)

print(f"wrote {out}  ({len(review)} venues)")
print(f"still missing a city : {review['city'].isna().sum()}")
print(f"multiple cities      : {(review['distinct_cities'] > 1).sum()}")
print(f"name overlaps        : {review['name_overlap'].notna().sum()}")

print("\ntop 15:")
print(review.head(15)[["matches_played", "city", "likely_country",
                       "distinct_cities"]].to_string())

print("\nname overlaps:")
print(review[review["name_overlap"].notna()][
    ["matches_played", "city", "name_overlap"]].head(20).to_string())

wrote d:\Tanay Nagpal\Labmentrix Internship\cricbuzz_livestats\data\processed\venues_review.csv  (382 venues)
still missing a city : 9
multiple cities      : 6
name overlaps        : 242

top 15:
                                                             matches_played       city likely_country  distinct_cities
venue                                                                                                                 
Harare Sports Club                                                      127     Harare       Zimbabwe                1
Shere Bangla National Stadium                                            85     Mirpur     Bangladesh                2
Dubai International Cricket Stadium                                      59      Dubai       Pakistan                1
Rangiri Dambulla International Stadium                                   51   Dambulla      Sri Lanka                1
R Premadasa Stadium                                                      49    Colombo    

In [33]:
def geo_candidates(city) -> list:
    """Countries containing a city of this name, largest first.

    Uses pd.isna rather than truthiness: a missing city arrives as NaN,
    which is a float and is truthy, so `if not city` would let it through.
    """
    if pd.isna(city) or not str(city).strip():
        return []

    name = str(city).lower()
    hits = [c for c in GEO_CITIES.values() if c["name"].lower() == name]
    hits.sort(key=lambda c: -c["population"])
    return [GEO_COUNTRIES[h["countrycode"]]["name"] for h in hits]

In [37]:
import geonamescache

gc = geonamescache.GeonamesCache()
GEO_CITIES = gc.get_cities()
GEO_COUNTRIES = gc.get_countries()

# Cricket names teams, not countries. England, Scotland and Ireland are all
# "United Kingdom" to a geographer, so the team signal disambiguates them.
UK_TEAMS = {"England", "Scotland", "Ireland"}


def geo_candidates(city) -> list:
    """Countries containing a city of this name, largest first.

    Uses pd.isna rather than truthiness: a missing city arrives as NaN,
    which is a float and is truthy, so `if not city` would let it through
    and then fail on .lower().
    """
    if pd.isna(city) or not str(city).strip():
        return []

    name = str(city).lower()
    hits = [c for c in GEO_CITIES.values() if c["name"].lower() == name]
    hits.sort(key=lambda c: -c["population"])
    return [GEO_COUNTRIES[h["countrycode"]]["name"] for h in hits]


def resolve_country(city, team_country):
    """Combine the geographic and cricketing signals.

    geonames is authoritative on where a city is but ambiguous on common
    names. The team signal is authoritative on whose cricket is played
    there but wrong at neutral venues. Each covers the other's blind spot.
    """
    candidates = geo_candidates(city)

    if not candidates:
        return team_country, "geo-miss"

    if team_country in candidates:
        return team_country, "agree"

    if team_country in UK_TEAMS and "United Kingdom" in candidates:
        return team_country, "uk"

    if len(set(candidates)) == 1:
        return candidates[0], "geo-only"

    return candidates[0], "REVIEW"


resolved = review.copy()
results = [resolve_country(r["city"], r["likely_country"])
           for _, r in resolved.iterrows()]

resolved["country"] = [r[0] for r in results]
resolved["how"] = [r[1] for r in results]

print("resolution method:")
for how, n in resolved["how"].value_counts().items():
    print(f"   {n:>4}  {how}")

print("\nwhere geography overruled the teams (neutral venues):")
print(resolved[resolved["how"] == "geo-only"][
    ["matches_played", "city", "likely_country", "country"]]
    .sort_values("matches_played", ascending=False).head(15).to_string())

print("\nneeds your eyes:")
print(resolved[resolved["how"] == "REVIEW"][
    ["matches_played", "city", "likely_country", "country"]]
    .sort_values("matches_played", ascending=False).head(20).to_string())

resolved.to_csv(PROCESSED / "venues_review.csv")
print("\nrewrote venues_review.csv")

resolution method:
    172  agree
     86  geo-only
     61  geo-miss
     39  uk
     24  REVIEW

where geography overruled the teams (neutral venues):
                                           matches_played          city            likely_country                country
venue                                                                                                                   
Dubai International Cricket Stadium                    59         Dubai                  Pakistan   United Arab Emirates
Sheikh Zayed Stadium                                   37     Abu Dhabi                  Pakistan   United Arab Emirates
Sharjah Cricket Stadium                                32       Sharjah                  Pakistan   United Arab Emirates
Sabina Park, Kingston                                  19       Jamaica               West Indies          United States
VRA Ground, Amstelveen                                 18    Amstelveen               Netherlands        The Netherlands


In [38]:
resolved["confidence"] = resolved["how"].map({
    "agree": "high", "uk": "high",
    "geo-only": "medium", "geo-miss": "medium",
    "REVIEW": "LOW",
})

priority = resolved.sort_values(
    ["confidence", "matches_played"],
    ascending=[True, False],
)[["matches_played", "city", "likely_country", "country",
   "how", "confidence"]]

out = PROCESSED / "venues_review.csv"
priority.to_csv(out)

covered = resolved.nlargest(60, "matches_played")["matches_played"].sum()
print(f"top 60 venues cover {covered:,} of {resolved['matches_played'].sum():,} "
      f"matches ({covered / resolved['matches_played'].sum():.0%})")
print(f"\nwrote {out}")

top 60 venues cover 1,771 of 3,182 matches (56%)

wrote d:\Tanay Nagpal\Labmentrix Internship\cricbuzz_livestats\data\processed\venues_review.csv


In [40]:
import pandas as pd
from pathlib import Path

# Find the project root by walking up until we see data/processed.
# Notebooks run from notebooks/, scripts run from the repo root,
# so a hardcoded relative path breaks in one place or the other.
root = Path.cwd()
while not (root / "data" / "processed").exists() and root != root.parent:
    root = root.parent

PROCESSED = root / "data" / "processed"

print("notebook is running from:", Path.cwd())
print("project root found at   :", root)
print("processed folder exists :", PROCESSED.exists())

if not PROCESSED.exists():
    print("\nCould not find data/processed anywhere above the notebook.")
else:
    print("\nfiles present:")
    for f in sorted(PROCESSED.iterdir()):
        print(f"   {f.name:25s} {f.stat().st_size / 1_048_576:6.1f} MB")

    matches = pd.read_parquet(PROCESSED / "matches.parquet")

    cg = matches[matches["venue"] == "County Ground"].copy()

    print(f"\nmatches at 'County Ground': {len(cg)}")
    print("\ncity values on these rows:")
    print(cg["city"].value_counts(dropna=False))

    print(f"\nall columns available: {list(matches.columns)}")

    wanted = ["match_id", "start_date", "season", "city",
              "team1", "team2", "event_name", "match_type"]
    cols = [c for c in wanted if c in cg.columns]

    sort_col = "start_date" if "start_date" in cols else cols[0]
    print(f"\n--- the {len(cg)} matches ---")
    print(cg[cols].sort_values(sort_col).to_string(index=False))

notebook is running from: d:\Tanay Nagpal\Labmentrix Internship\cricbuzz_livestats\notebooks
project root found at   : d:\Tanay Nagpal\Labmentrix Internship\cricbuzz_livestats
processed folder exists : True

files present:
   batting.parquet              0.7 MB
   bowling.parquet              0.3 MB
   matches.parquet              0.1 MB
   people.parquet               0.1 MB
   squads.parquet               0.3 MB
   venues_review.csv            0.0 MB

matches at 'County Ground': 41

city values on these rows:
city
Bristol       17
Taunton       11
Derby          8
Hove           3
Chelmsford     2
Name: count, dtype: int64

all columns available: ['match_id', 'match_format', 'match_type_number', 'gender', 'team_type', 'season', 'match_date', 'end_date', 'venue', 'city', 'team1', 'team2', 'toss_winner', 'toss_decision', 'winner', 'victory_type', 'victory_margin', 'victory_method', 'player_of_match', 'series_name', 'event_match_number', 'balls_per_over', 'scheduled_overs']

--- the 41 

In [41]:
# 1. Are women's matches mixed into this archive?
#    The 2017 Derby / Taunton / Bristol fixtures look like the
#    Women's World Cup, not the men's Champions Trophy.
print("matches by gender:")
print(matches["gender"].value_counts(dropna=False))
print("\nmatches by team_type:")
print(matches["team_type"].value_counts(dropna=False))

# 2. How many venues do we actually have, keyed correctly?
print(f"\ndistinct venue strings       : {matches['venue'].nunique()}")
print(f"distinct (venue, city) pairs : "
      f"{matches.groupby(['venue', 'city'], dropna=False).ngroups}")
print(f"rows with a missing city     : {matches['city'].isna().sum()}")

# 3. Which venue names cover more than one city?
multi = (matches.groupby("venue")["city"]
                .nunique(dropna=False)
                .sort_values(ascending=False))
multi = multi[multi > 1]
print(f"\nvenue names spanning more than one city: {len(multi)}")
print(multi.head(20).to_string())

matches by gender:
gender
male      2571
female     611
Name: count, dtype: int64

matches by team_type:
team_type
international    3182
Name: count, dtype: int64

distinct venue strings       : 382
distinct (venue, city) pairs : 406
rows with a missing city     : 306

venue names spanning more than one city: 20
venue
County Ground                                   5
Nehru Stadium                                   3
Dubai International Cricket Stadium             2
Himachal Pradesh Cricket Association Stadium    2
Bulawayo Athletic Club                          2
Galle International Stadium                     2
Harare Sports Club                              2
Maple Leaf North-West Ground                    2
Melbourne Cricket Ground                        2
Multan Cricket Stadium                          2
Pallekele International Cricket Stadium         2
Perth Stadium                                   2
Rangiri Dambulla International Stadium          2
Rawalpindi Cricket Stadium    

In [42]:
import pandas as pd
from pathlib import Path

root = Path.cwd()
while not (root / "data" / "processed").exists() and root != root.parent:
    root = root.parent
PROCESSED = root / "data" / "processed"

orig = pd.read_csv(PROCESSED / "venues_review.csv")
edit = pd.read_csv(PROCESSED / "venues_manual.csv")

print(f"original rows : {len(orig)}")
print(f"edited rows   : {len(edit)}")

# matches_played was never edited, so its sequence is a fingerprint.
# If it still matches row for row, the rows never moved and we can
# safely pair them up. If it doesn't, no join is trustworthy.
aligned = (len(orig) == len(edit)
           and orig["matches_played"].equals(edit["matches_played"]))
print(f"\nrow order intact: {aligned}")

if not aligned:
    print("STOP - rows moved. Don't go further, tell me.")
else:
    edit["raw_venue"] = orig["venue"]   # recover the original string

    for col in ["venue", "city", "country"]:
        a = orig[col].fillna("<blank>")
        b = edit[col].fillna("<blank>")
        print(f"  {col:8s} changed on {(a != b).sum():3d} rows")

    renamed = edit[orig["venue"] != edit["venue"]]
    print(f"\nsample of renamed venues (raw -> your version):")
    print(renamed[["raw_venue", "venue", "city", "country"]]
          .head(15).to_string(index=False))

original rows : 382
edited rows   : 382

row order intact: False
STOP - rows moved. Don't go further, tell me.


In [43]:
# 1. Are the numbers even stored the same way?
print("dtypes:")
print(f"  original: {orig['matches_played'].dtype}")
print(f"  edited  : {edit['matches_played'].dtype}")

# 2. Same values overall, just in a different order?
same_multiset = (sorted(orig["matches_played"].tolist())
                 == sorted(edit["matches_played"].tolist()))
print(f"\nsame set of values, possibly reordered: {same_multiset}")

# 3. Where does it first diverge?
a = orig["matches_played"].reset_index(drop=True)
b = edit["matches_played"].reset_index(drop=True)
diff = a.compare(b) if len(a) == len(b) else None
print(f"\nrows where matches_played differs: {(a != b).sum()}")
print("\nfirst 10 divergences (orig vs edited):")
print(diff.head(10).to_string() if diff is not None else "length mismatch")

# 4. Eyeball the top of each file
print("\n--- original, first 8 rows ---")
print(orig[["venue", "matches_played", "confidence"]].head(8).to_string())
print("\n--- edited, first 8 rows ---")
print(edit[["venue", "matches_played", "confidence"]].head(8).to_string())

dtypes:
  original: int64
  edited  : int64

same set of values, possibly reordered: True

rows where matches_played differs: 140

first 10 divergences (orig vs edited):
     self  other
235  85.0    1.0
236  59.0    1.0
237  40.0    1.0
238  37.0    1.0
239  32.0    1.0
240  28.0    1.0
241  21.0    1.0
242  20.0    1.0
243  19.0    1.0
244  18.0    1.0

--- original, first 8 rows ---
                                                  venue  matches_played confidence
0                      Queen's Park Oval, Port of Spain              22        LOW
1                                Hazelaarweg, Rotterdam               8        LOW
2                                           P Sara Oval               8        LOW
3                          John Davies Oval, Queenstown               6        LOW
4                        Sabina Park, Kingston, Jamaica               5        LOW
5                             County Ground, Chelmsford               4        LOW
6  Rajiv Gandhi International 

In [44]:
# Hypothesis: your rule was "keep everything before the first comma".
rule_clean = orig["venue"].str.split(",").str[0].str.strip()
his_clean = edit["venue"].str.strip()

print(f"distinct names, comma-split rule : {rule_clean.nunique()}")
print(f"distinct names, your version     : {his_clean.nunique()}")

only_rule = sorted(set(rule_clean) - set(his_clean))
only_his = sorted(set(his_clean) - set(rule_clean))

print(f"\nnames the rule makes but you didn't ({len(only_rule)}):")
print(only_rule[:25])

print(f"\nnames you made but the rule didn't ({len(only_his)}):")
print(only_his[:25])

# Where you gave the same cleaned venue more than one country,
# we need to know - that's the join breaking.
chk = edit.groupby(his_clean)["country"].nunique()
bad = chk[chk > 1]
print(f"\ncleaned names where you gave conflicting countries: {len(bad)}")
if len(bad):
    print(bad.to_string())

# How much city/country info is sitting in the raw strings already?
parts = orig["venue"].str.split(",")
print(f"\nraw strings with 2 comma-parts (venue, city)        : {(parts.str.len() == 2).sum()}")
print(f"raw strings with 3 comma-parts (venue, city, country): {(parts.str.len() == 3).sum()}")
print(f"raw strings with no comma at all                     : {(parts.str.len() == 1).sum()}")

distinct names, comma-split rule : 255
distinct names, your version     : 255

names the rule makes but you didn't (0):
[]

names you made but the rule didn't (0):
[]

cleaned names where you gave conflicting countries: 0

raw strings with 2 comma-parts (venue, city)        : 169
raw strings with 3 comma-parts (venue, city, country): 25
raw strings with no comma at all                     : 188


In [45]:
matches = pd.read_parquet(PROCESSED / "matches.parquet")

# 1. Your cleaning rule, applied in code.
matches["venue_clean"] = matches["venue"].str.split(",").str[0].str.strip()

# 2. City: three sources, best first.
#    (a) the match record's own city field - most trustworthy
#    (b) the second comma-part of the venue string
#    (c) whatever you typed in the spreadsheet
city_from_string = matches["venue"].str.split(",").str[1].str.strip()

his = edit.assign(v=edit["venue"].str.strip())
his_city = (his.dropna(subset=["city"])
               .drop_duplicates("v")
               .set_index("v")["city"])
his_country = his.drop_duplicates("v").set_index("v")["country"]

matches["city_final"] = (matches["city"]
                         .fillna(city_from_string)
                         .fillna(matches["venue_clean"].map(his_city)))

print(f"matches with no city, before : {matches['city'].isna().sum()}")
print(f"matches with no city, after  : {matches['city_final'].isna().sum()}")

# 3. Country, from your spreadsheet, joined on the cleaned name.
matches["venue_country"] = matches["venue_clean"].map(his_country)
print(f"matches with no country      : {matches['venue_country'].isna().sum()}")

# 4. The venue dimension at its true grain: one row per ground,
#    not one row per spelling.
dim_venue = (matches
             .groupby(["venue_clean", "city_final", "venue_country"], dropna=False)
             .agg(matches_played=("match_id", "size"))
             .reset_index()
             .sort_values("matches_played", ascending=False))

print(f"\nraw venue strings      : {matches['venue'].nunique()}")
print(f"cleaned venue names    : {matches['venue_clean'].nunique()}")
print(f"dim_venue rows (name+city): {len(dim_venue)}")

print("\n--- top 25 venues by matches hosted ---")
print(dim_venue.head(25).to_string(index=False))

print("\n--- any rows still missing city or country ---")
gaps = dim_venue[dim_venue["city_final"].isna() | dim_venue["venue_country"].isna()]
print(f"{len(gaps)} rows")
print(gaps.to_string(index=False) if len(gaps) else "none")

matches with no city, before : 306
matches with no city, after  : 0
matches with no country      : 0

raw venue strings      : 382
cleaned venue names    : 255
dim_venue rows (name+city): 280

--- top 25 venues by matches hosted ---
                                      venue_clean  city_final        venue_country  matches_played
                               Harare Sports Club      Harare             Zimbabwe             127
                              R Premadasa Stadium     Colombo            Sri Lanka              94
                    Shere Bangla National Stadium      Mirpur           Bangladesh              83
              Dubai International Cricket Stadium       Dubai United Arab Emirates              59
                               Queens Sports Club    Bulawayo             Zimbabwe              59
           Rangiri Dambulla International Stadium    Dambulla            Sri Lanka              51
                            Sydney Cricket Ground      Sydney            A

In [46]:
import re

# A. Which cleaned names show up with more than one city?
multi = dim_venue.groupby("venue_clean").filter(lambda g: len(g) > 1)
print(f"names appearing with more than one city: {multi['venue_clean'].nunique()}")
print(f"extra rows they create: {len(multi) - multi['venue_clean'].nunique()}\n")
print(multi.sort_values(["venue_clean", "matches_played"],
                        ascending=[True, False]).to_string(index=False))

# B. Names that differ only in punctuation or spacing.
#    "R Premadasa Stadium" and "R.Premadasa Stadium" are one ground,
#    but the comma rule can't see that - it only cuts at commas.
def squash(s):
    return re.sub(r"[^a-z0-9]", "", str(s).lower())

tmp = dim_venue.assign(key=dim_venue["venue_clean"].map(squash))
variants = tmp.groupby("key")["venue_clean"].nunique()
variants = variants[variants > 1]

print(f"\n\nname variants differing only by punctuation/spacing: {len(variants)}")
for k in variants.index:
    group = tmp.loc[tmp["key"] == k]
    print(f"\n  {sorted(group['venue_clean'].unique())}")
    print(group[["venue_clean", "city_final", "matches_played"]].to_string(index=False))

names appearing with more than one city: 17
extra rows they create: 25

                                 venue_clean     city_final                    venue_country  matches_played
                           Arnos Vale Ground     St Vincent Saint Vincent and the Grenadines               8
                           Arnos Vale Ground      Kingstown Saint Vincent and the Grenadines               3
                               County Ground        Bristol                          England              25
                               County Ground          Derby                          England              12
                               County Ground        Taunton                          England              11
                               County Ground      Worcester                          England               8
                               County Ground     Chelmsford                          England               6
                               County Ground           H

In [47]:
# Three failure modes, one lookup each. Written out so a reader can
# see the reasoning rather than trusting a magic dictionary.

CITY_ALIASES = {
    # (1) an island or country written where a city belongs
    "Jamaica": "Kingston",
    "Barbados": "Bridgetown",
    "Trinidad": "Port of Spain",
    "Grenada": "St George's",
    "Guyana": "Providence",
    "Antigua": "North Sound",
    "St Kitts": "Basseterre",
    "St Lucia": "Gros Islet",
    "St Vincent": "Kingstown",

    # (2) real-world renames - we keep the current official name
    "Bangalore": "Bengaluru",
    "Port Elizabeth": "Gqeberha",
    "Chittagong": "Chattogram",
    "Dharmasala": "Dharamsala",     # spelling drift, not a rename

    # (3) a neighbourhood recorded instead of its city
    "Mirpur": "Dhaka",
    "Brighton": "Hove",             # the ground is always called Hove
}

# Punctuation drift the comma rule couldn't see.
VENUE_NAME_ALIASES = {
    "M.Chinnaswamy Stadium": "M Chinnaswamy Stadium",
    "R.Premadasa Stadium": "R Premadasa Stadium",
}

matches["city_final"] = matches["city_final"].replace(CITY_ALIASES)
matches["venue_clean"] = matches["venue_clean"].replace(VENUE_NAME_ALIASES)

dim_venue = (matches
             .groupby(["venue_clean", "city_final", "venue_country"], dropna=False)
             .agg(matches_played=("match_id", "size"))
             .reset_index()
             .sort_values("matches_played", ascending=False))

print(f"dim_venue rows now: {len(dim_venue)}   (was 280)")

# Only County Ground and Nehru Stadium should still split.
still = dim_venue.groupby("venue_clean").filter(lambda g: len(g) > 1)
print(f"\nnames still spanning more than one city: "
      f"{still['venue_clean'].nunique()}")
print(still.sort_values(["venue_clean", "matches_played"],
                        ascending=[True, False]).to_string(index=False))

# A ground whose name isn't unique needs the city shown alongside it,
# or a dropdown lists "County Ground" seven times.
ambiguous = set(dim_venue["venue_clean"]
                .value_counts()
                .loc[lambda s: s > 1].index)

dim_venue["venue_display"] = [
    f"{v}, {c}" if v in ambiguous else v
    for v, c in zip(dim_venue["venue_clean"], dim_venue["city_final"])
]

print("\n--- top 15, as they'll appear in the app ---")
print(dim_venue[["venue_display", "venue_country", "matches_played"]]
      .head(15).to_string(index=False))

dim_venue rows now: 263   (was 280)

names still spanning more than one city: 3
                 venue_clean  city_final venue_country  matches_played
               County Ground     Bristol       England              25
               County Ground       Derby       England              12
               County Ground     Taunton       England              11
               County Ground   Worcester       England               8
               County Ground  Chelmsford       England               6
               County Ground        Hove       England               4
               County Ground Northampton       England               2
Maple Leaf North-West Ground   King City        Canada              17
Maple Leaf North-West Ground     Toronto        Canada               1
               Nehru Stadium       Kochi         India               4
               Nehru Stadium    Guwahati         India               3
               Nehru Stadium      Margao         India              

In [48]:
# A global city alias would be wrong here (Toronto is a real city),
# so this override is keyed on the venue AND the city together.
# Narrow rules for narrow problems.
VENUE_CITY_OVERRIDES = {
    ("Maple Leaf North-West Ground", "Toronto"): "King City",
}

matches["city_final"] = [
    VENUE_CITY_OVERRIDES.get((v, c), c)
    for v, c in zip(matches["venue_clean"], matches["city_final"])
]

dim_venue = (matches
             .groupby(["venue_clean", "city_final", "venue_country"], dropna=False)
             .agg(matches_played=("match_id", "size"))
             .reset_index()
             .sort_values("matches_played", ascending=False))

print(f"dim_venue rows: {len(dim_venue)}")

# --- home_nation ---
# Your country column says where the ground IS. This says which
# international side calls it home. Ten Caribbean nations, one team.
WEST_INDIES = {
    "Antigua and Barbuda", "Barbados", "Dominica", "Grenada", "Guyana",
    "Jamaica", "Saint Kitts and Nevis", "Saint Lucia",
    "Saint Vincent and the Grenadines", "Trinidad and Tobago",
}

dim_venue["home_nation"] = dim_venue["venue_country"].apply(
    lambda c: "West Indies" if c in WEST_INDIES else c
)

# Critical check: do those names match the team names in the data?
# If they don't, the home-advantage query silently returns nothing.
teams = sorted(t for t in pd.unique(matches[["team1", "team2"]].values.ravel())
               if pd.notna(t))
print(f"\ndistinct teams in the archive: {len(teams)}")
print(teams)

unmatched = sorted(set(dim_venue["home_nation"]) - set(teams))
print(f"\nhome_nation values with no matching team ({len(unmatched)}):")
print(unmatched)

dim_venue rows: 262

distinct teams in the archive: 28
['Africa XI', 'Asia XI', 'Australia', 'Bangladesh', 'Bermuda', 'Canada', 'England', 'Hong Kong', 'ICC World XI', 'India', 'Ireland', 'Jersey', 'Kenya', 'Namibia', 'Nepal', 'Netherlands', 'New Zealand', 'Oman', 'Pakistan', 'Papua New Guinea', 'Scotland', 'South Africa', 'Sri Lanka', 'Thailand', 'United Arab Emirates', 'United States of America', 'West Indies', 'Zimbabwe']

home_nation values with no matching team (4):
['Malaysia', 'Spain', 'The Netherlands', 'United States']


In [49]:
# Match the team spellings - home_nation is a join key, not a label.
HOME_NATION_FIXES = {
    "The Netherlands": "Netherlands",
    "United States": "United States of America",
}
dim_venue["home_nation"] = dim_venue["home_nation"].replace(HOME_NATION_FIXES)

# Countries that host but field no side. NULL is the honest value -
# better than a string that looks joinable and never joins.
dim_venue.loc[dim_venue["home_nation"].isin({"Spain", "Malaysia"}),
              "home_nation"] = None

still = sorted(set(dim_venue["home_nation"].dropna()) - set(teams))
print(f"home_nation values still unmatched: {still}")

# --- surrogate key ---
# Sorted first, so re-running the ETL always produces the same ids.
# An id that shuffles between runs breaks every foreign key.
dim_venue = (dim_venue
             .sort_values(["venue_clean", "city_final"])
             .reset_index(drop=True))
dim_venue.insert(0, "venue_id", range(1, len(dim_venue) + 1))

# Display name: show the city only when the name alone is ambiguous.
ambiguous = set(dim_venue["venue_clean"]
                .value_counts()
                .loc[lambda s: s > 1].index)
dim_venue["venue_display"] = [
    f"{v}, {c}" if v in ambiguous else v
    for v, c in zip(dim_venue["venue_clean"], dim_venue["city_final"])
]

# --- attach the key back to every match ---
lookup = dim_venue.set_index(["venue_clean", "city_final"])["venue_id"]
matches["venue_id"] = pd.MultiIndex.from_arrays(
    [matches["venue_clean"], matches["city_final"]]
).map(lookup)

print(f"\ndim_venue rows            : {len(dim_venue)}")
print(f"matches with no venue_id  : {matches['venue_id'].isna().sum()}")
print(f"venue_ids actually used   : {matches['venue_id'].nunique()}")

dim_venue.to_parquet(PROCESSED / "dim_venue.parquet", index=False)
print(f"\nsaved -> {PROCESSED / 'dim_venue.parquet'}")

print("\n--- sample ---")
print(dim_venue[["venue_id", "venue_display", "city_final",
                 "venue_country", "home_nation", "matches_played"]]
      .sort_values("matches_played", ascending=False)
      .head(12).to_string(index=False))

home_nation values still unmatched: []

dim_venue rows            : 262
matches with no venue_id  : 0
venue_ids actually used   : 262

saved -> d:\Tanay Nagpal\Labmentrix Internship\cricbuzz_livestats\data\processed\dim_venue.parquet

--- sample ---
 venue_id                                     venue_display city_final        venue_country          home_nation  matches_played
       88                                Harare Sports Club     Harare             Zimbabwe             Zimbabwe             127
      206                     Shere Bangla National Stadium      Dhaka           Bangladesh           Bangladesh             127
      176                               R Premadasa Stadium    Colombo            Sri Lanka            Sri Lanka             107
       66               Dubai International Cricket Stadium      Dubai United Arab Emirates United Arab Emirates              59
      174                                Queens Sports Club   Bulawayo             Zimbabwe             Z

In [51]:
import re
import requests
from io import StringIO

URL = "https://en.wikipedia.org/wiki/List_of_cricket_grounds_by_capacity"

# Wikipedia blocks requests that don't identify themselves, so we send
# a User-Agent. This is the polite way to scrape - it tells their
# servers who is asking.
resp = requests.get(
    URL,
    headers={"User-Agent": "cricbuzz-livestats/1.0 (student project)"},
    timeout=30,
)
resp.raise_for_status()
print(f"fetched {len(resp.text):,} characters")

# pandas can read every <table> on a page in one go.
tables = pd.read_html(StringIO(resp.text))
print(f"tables found on the page: {len(tables)}")

# The page has several tables (active grounds, proposed, closed).
# Keep only the ones that have both a ground name and a capacity.
wanted = []
for i, t in enumerate(tables):
    cols = [str(c) for c in t.columns]
    has_name = any("Ground" in c or "Stadium" in c for c in cols)
    has_cap = any("Capacity" in c for c in cols)
    if has_name and has_cap:
        print(f"   table {i}: {t.shape[0]} rows  {cols}")
        wanted.append(t)

wiki = pd.concat(wanted, ignore_index=True)
print(f"\ncombined rows: {len(wiki)}")
print(f"columns: {list(wiki.columns)}")
print("\n--- first 8 rows ---")
print(wiki.head(8).to_string())

fetched 642,947 characters
tables found on the page: 9
   table 0: 2 rows  ['Ground', 'Capacity', 'City', 'Country', 'Home teams', 'Image']
   table 1: 11 rows  ['Ground', 'Capacity', 'City', 'Country', 'Home teams', 'Image']
   table 2: 16 rows  ['Ground', 'Capacity', 'City', 'Country', 'Home teams', 'Image']
   table 3: 22 rows  ['Ground', 'Capacity', 'City', 'Country', 'Home teams', 'Image']
   table 4: 35 rows  ['Ground', 'Capacity', 'City', 'Country', 'Home teams', 'Image']
   table 5: 58 rows  ['Ground', 'Capacity', 'City', 'Country', 'Home teams', 'Image']
   table 6: 59 rows  ['Ground', 'Capacity', 'City', 'Country', 'Home teams']
   table 7: 27 rows  ['Ground', 'Capacity', 'Country', 'City', 'Home team', 'Estimated completion date']
   table 8: 19 rows  ['Ground', 'Capacity', 'City', 'Country', 'Home team', 'Closed (as a cricket ground)', 'Cause']

combined rows: 249
columns: ['Ground', 'Capacity', 'City', 'Country', 'Home teams', 'Image', 'Home team', 'Estimated completion da

In [52]:
def squash(s):
    """Lowercase, strip everything that isn't a letter or digit.
    Makes 'St George's Park' and 'St Georges Park' the same key."""
    return re.sub(r"[^a-z0-9]", "", str(s).lower())


def clean_capacity(v):
    """'132,000[1][2][3]' -> 132000.
    Wikipedia glues footnote markers onto the number and is
    inconsistent about thousands separators, so strip to digits."""
    s = str(v).split("[")[0]
    s = re.sub(r"[^0-9]", "", s)
    return int(s) if s else None


keep = []
for i, t in enumerate(tables):
    cols = [str(c) for c in t.columns]
    if not (any("Ground" in c for c in cols) and any("Capacity" in c for c in cols)):
        continue
    if any("Estimated completion" in c for c in cols):
        print(f"   skipping table {i} ({len(t)} rows) - not built yet")
        continue
    label = "closed" if any("Closed" in c for c in cols) else "active"
    print(f"   keeping  table {i} ({len(t)} rows) - {label}")
    keep.append(t[["Ground", "Capacity", "City", "Country"]])

wiki = pd.concat(keep, ignore_index=True)
wiki["capacity"] = wiki["Capacity"].map(clean_capacity)
print(f"\nwiki rows: {len(wiki)}   capacity parsed on {wiki['capacity'].notna().sum()}")

wiki["gkey"] = wiki["Ground"].map(squash)
wiki["ckey"] = wiki["City"].map(squash)
dim_venue["gkey"] = dim_venue["venue_clean"].map(squash)
dim_venue["ckey"] = dim_venue["city_final"].map(squash)

# Pass 1 - ground name AND city agree. Strongest match.
w1 = wiki.dropna(subset=["capacity"]).drop_duplicates(["gkey", "ckey"])
w1 = w1.set_index(["gkey", "ckey"])["capacity"]
cap1 = pd.Series(
    pd.MultiIndex.from_arrays([dim_venue["gkey"], dim_venue["ckey"]]).map(w1),
    index=dim_venue.index,
)

# Pass 2 - ground name alone, but ONLY where that name appears once
# on the Wikipedia side. A name that appears twice is ambiguous and
# we'd rather have no capacity than the wrong one.
counts = wiki["gkey"].value_counts()
unique_names = counts[counts == 1].index
w2 = (wiki[wiki["gkey"].isin(unique_names)]
      .dropna(subset=["capacity"])
      .set_index("gkey")["capacity"])
cap2 = dim_venue["gkey"].map(w2)

dim_venue["capacity"] = cap1.fillna(cap2)

matched = dim_venue["capacity"].notna()
total_m = dim_venue["matches_played"].sum()
print(f"\nvenues with a capacity : {matched.sum()} of {len(dim_venue)}")
print(f"matches covered        : {dim_venue.loc[matched, 'matches_played'].sum():,} of {total_m:,}"
      f"  ({dim_venue.loc[matched, 'matches_played'].sum() / total_m:.0%})")

print("\n--- biggest venues still missing a capacity ---")
print(dim_venue.loc[~matched, ["venue_display", "city_final",
                               "venue_country", "matches_played"]]
      .sort_values("matches_played", ascending=False)
      .head(20).to_string(index=False))

   keeping  table 0 (2 rows) - active
   keeping  table 1 (11 rows) - active
   keeping  table 2 (16 rows) - active
   keeping  table 3 (22 rows) - active
   keeping  table 4 (35 rows) - active
   keeping  table 5 (58 rows) - active
   keeping  table 6 (59 rows) - active
   skipping table 7 (27 rows) - not built yet
   keeping  table 8 (19 rows) - closed

wiki rows: 222   capacity parsed on 218

venues with a capacity : 109 of 262
matches covered        : 1,792 of 3,182  (56%)

--- biggest venues still missing a capacity ---
                                          venue_display      city_final         venue_country  matches_played
                          Shere Bangla National Stadium           Dhaka            Bangladesh             127
                                        Kennington Oval          London               England              48
                                        SuperSport Park       Centurion          South Africa              46
                             

In [53]:
from difflib import get_close_matches, SequenceMatcher

wiki_lookup = (wiki.dropna(subset=["capacity"])
                   .drop_duplicates("gkey")
                   .set_index("gkey"))
wiki_keys = list(wiki_lookup.index)

ACCEPT = 0.85   # similarity needed to auto-accept

accepted, review = [], []

for idx, r in dim_venue[~matched].sort_values(
        "matches_played", ascending=False).iterrows():

    hits = get_close_matches(r["gkey"], wiki_keys, n=2, cutoff=0.55)
    for h in hits:
        w = wiki_lookup.loc[h]
        score = SequenceMatcher(None, r["gkey"], h).ratio()
        row = {
            "idx": idx,
            "our_venue": r["venue_display"],
            "matches": r["matches_played"],
            "wiki_ground": w["Ground"],
            "wiki_country": w["Country"],
            "our_country": r["venue_country"],
            "capacity": int(w["capacity"]),
            "score": round(score, 2),
        }
        # Similarity alone is not enough - the country must agree too.
        if score >= ACCEPT and str(w["Country"]).strip() == str(r["venue_country"]).strip():
            accepted.append(row)
            break
        review.append(row)

acc = pd.DataFrame(accepted)
rev = pd.DataFrame(review)

print(f"auto-accepted: {len(acc)} venues")
if len(acc):
    print(acc[["our_venue", "wiki_ground", "capacity", "matches", "score"]]
          .to_string(index=False))

print(f"\n\nneeds a human look: {rev['our_venue'].nunique() if len(rev) else 0} venues")
if len(rev):
    print(rev[["our_venue", "our_country", "wiki_ground", "wiki_country",
               "capacity", "matches", "score"]]
          .sort_values("matches", ascending=False)
          .head(30).to_string(index=False))

auto-accepted: 19 venues
                                                 our_venue                                                  wiki_ground  capacity  matches  score
                                    County Ground, Bristol                                                County Ground     12500       25   1.00
                                     New Wanderers Stadium                                            Wanderers Stadium     34000       24   0.91
                             Zahur Ahmed Chowdhury Stadium                                Zohur Ahmed Chowdhury Stadium     22000       17   0.96
                                     The Wanderers Stadium                                            Wanderers Stadium     34000       17   0.91
                        Punjab Cricket Association Stadium                            Assam Cricket Association Stadium     46000       14   0.85
                                      County Ground, Derby                                         

In [54]:
from difflib import SequenceMatcher

# Look for duplicates INSIDE dim_venue. Requiring the same city and
# country first is what makes this safe - it removes the entire class
# of false pairs that the Wikipedia match fell into.
dv = dim_venue.reset_index(drop=True)
pairs = []

for i in range(len(dv)):
    for j in range(i + 1, len(dv)):
        a, b = dv.loc[i], dv.loc[j]
        if a["city_final"] != b["city_final"]:
            continue
        if a["venue_country"] != b["venue_country"]:
            continue
        score = SequenceMatcher(None, a["gkey"], b["gkey"]).ratio()
        if score >= 0.55:
            pairs.append({
                "city": a["city_final"],
                "venue_a": a["venue_clean"],
                "m_a": a["matches_played"],
                "venue_b": b["venue_clean"],
                "m_b": b["matches_played"],
                "score": round(score, 2),
            })

dup = pd.DataFrame(pairs)
print(f"candidate duplicate pairs in dim_venue: {len(dup)}")
if len(dup):
    print(dup.sort_values("score", ascending=False).to_string(index=False))

# Also worth seeing: any city hosting several differently-named grounds
# that we should eyeball regardless of similarity.
multi_city = (dv.groupby(["city_final", "venue_country"])
                .agg(grounds=("venue_clean", "nunique"),
                     matches=("matches_played", "sum"))
                .query("grounds > 1")
                .sort_values("matches", ascending=False))
print(f"\n\ncities with more than one ground: {len(multi_city)}")
print(multi_city.head(20).to_string())

candidate duplicate pairs in dim_venue: 52
         city                                                    venue_a  m_a                                                      venue_b  m_b  score
    Al Amarat    Al Amerat Cricket Ground Oman Cricket (Ministry Turf 1)   28      Al Amerat Cricket Ground Oman Cricket (Ministry Turf 2)    5   0.98
   Gros Islet                       Daren Sammy National Cricket Stadium    5                        Darren Sammy National Cricket Stadium    1   0.98
      Lucknow Bharat Ratna Shri Atal Bihari Vajpai Ekana Cricket Stadium    2 Bharat Ratna Shri Atal Bihari Vajpayee Ekana Cricket Stadium    8   0.96
   Chattogram                              Zahur Ahmed Chowdhury Stadium   17                                Zohur Ahmed Chowdhury Stadium    3   0.96
 Stellenbosch                                  Stellenbosch University 1    2                                    Stellenbosch University 2    1   0.96
 Johannesburg                                      

In [55]:
# Each entry: a name we saw -> the name we keep.
# Grouped by WHY they differ, so this reads as reasoning
# rather than as a magic lookup table.
VENUE_MERGES = {
    # --- sponsor names ---
    "The Royal & Sun Alliance County Ground": "County Ground",
    "The Cooper Associates County Ground": "County Ground",
    "De Beers Diamond Oval": "Diamond Oval",
    "Boland Bank Park": "Boland Park",
    "Sedgars Park": "Senwes Park",
    "AMI Stadium": "Jade Stadium",
    "Sky Stadium": "Westpac Stadium",
    "Choice Moosa Stadium": "Moosa Cricket Stadium",

    # --- official renames ---
    "Khettarama Stadium": "R Premadasa Stadium",
    "Punjab Cricket Association Stadium":
        "Punjab Cricket Association IS Bindra Stadium",
    "Sharjah Cricket Association Stadium": "Sharjah Cricket Stadium",
    "Narayanganj Osmani Stadium": "Khan Shaheb Osman Ali Stadium",

    # --- spelling ---
    "Darren Sammy National Cricket Stadium":
        "Daren Sammy National Cricket Stadium",
    "Bharat Ratna Shri Atal Bihari Vajpai Ekana Cricket Stadium":
        "Bharat Ratna Shri Atal Bihari Vajpayee Ekana Cricket Stadium",
    "Zohur Ahmed Chowdhury Stadium": "Zahur Ahmed Chowdhury Stadium",
    "Sher-e-Bangla National Cricket Stadium":
        "Shere Bangla National Stadium",
    "Vidarbha C.A. Ground": "Vidarbha Cricket Association Ground",

    # --- same name, extra or missing words ---
    "Sinhalese Sports Club": "Sinhalese Sports Club Ground",
    "Grange Cricket Club": "Grange Cricket Club Ground",
    "VRA Cricket Ground": "VRA Ground",
    "Sardar Patel (Gujarat) Stadium": "Sardar Patel Stadium",
    "ICC Global Cricket Academy": "ICC Academy",
    "New Wanderers Stadium": "The Wanderers Stadium",
    "Zayed Cricket Stadium": "Sheikh Zayed Stadium",
    "Dubai Sports City Cricket Stadium": "Dubai International Cricket Stadium",
    "Davies Park": "John Davies Oval",
}

before = matches["venue_clean"].nunique()
matches["venue_clean"] = matches["venue_clean"].replace(VENUE_MERGES)
print(f"distinct venue names: {before} -> {matches['venue_clean'].nunique()}")

dim_venue = (matches
             .groupby(["venue_clean", "city_final", "venue_country"], dropna=False)
             .agg(matches_played=("match_id", "size"))
             .reset_index()
             .sort_values(["venue_clean", "city_final"])
             .reset_index(drop=True))

print(f"dim_venue rows: {len(dim_venue)}   (was 262)")

# Sanity: the totals must not move. Merging changes how matches are
# grouped, never how many there are.
print(f"total matches: {dim_venue['matches_played'].sum():,} (must be 3,182)")

print("\n--- the merged grounds, with their new totals ---")
merged_names = sorted(set(VENUE_MERGES.values()))
print(dim_venue[dim_venue["venue_clean"].isin(merged_names)]
      [["venue_clean", "city_final", "matches_played"]]
      .sort_values("matches_played", ascending=False)
      .to_string(index=False))

distinct venue names: 253 -> 227
dim_venue rows: 236   (was 262)
total matches: 3,182 (must be 3,182)

--- the merged grounds, with their new totals ---
                                                 venue_clean    city_final  matches_played
                               Shere Bangla National Stadium         Dhaka             128
                                         R Premadasa Stadium       Colombo             108
                         Dubai International Cricket Stadium         Dubai              61
                                       The Wanderers Stadium  Johannesburg              41
                                        Sheikh Zayed Stadium     Abu Dhabi              40
                                     Sharjah Cricket Stadium       Sharjah              39
                                                 Senwes Park Potchefstroom              29
                                                 ICC Academy         Dubai              28
                            

In [56]:
# Rebuild home_nation (the dedup rebuilt dim_venue from scratch).
WEST_INDIES = {
    "Antigua and Barbuda", "Barbados", "Dominica", "Grenada", "Guyana",
    "Jamaica", "Saint Kitts and Nevis", "Saint Lucia",
    "Saint Vincent and the Grenadines", "Trinidad and Tobago",
}
HOME_FIXES = {"The Netherlands": "Netherlands",
              "United States": "United States of America"}

dim_venue["home_nation"] = (dim_venue["venue_country"]
    .apply(lambda c: "West Indies" if c in WEST_INDIES else c)
    .replace(HOME_FIXES))
dim_venue.loc[dim_venue["home_nation"].isin({"Spain", "Malaysia"}),
              "home_nation"] = None

# --- capacity, exact matches only ---
dim_venue["gkey"] = dim_venue["venue_clean"].map(squash)
dim_venue["ckey"] = dim_venue["city_final"].map(squash)

w1 = (wiki.dropna(subset=["capacity"])
          .drop_duplicates(["gkey", "ckey"])
          .set_index(["gkey", "ckey"])["capacity"])
cap1 = pd.Series(
    pd.MultiIndex.from_arrays([dim_venue["gkey"], dim_venue["ckey"]]).map(w1),
    index=dim_venue.index)

counts = wiki["gkey"].value_counts()
w2 = (wiki[wiki["gkey"].isin(counts[counts == 1].index)]
      .dropna(subset=["capacity"])
      .set_index("gkey")["capacity"])
dim_venue["capacity"] = cap1.fillna(dim_venue["gkey"].map(w2))

hit = dim_venue["capacity"].notna()
total = dim_venue["matches_played"].sum()
print(f"venues with capacity : {hit.sum()} of {len(dim_venue)}")
print(f"matches covered      : {dim_venue.loc[hit, 'matches_played'].sum():,} "
      f"of {total:,} ({dim_venue.loc[hit, 'matches_played'].sum() / total:.0%})")

# For the biggest misses, show the closest Wikipedia names as
# SUGGESTIONS ONLY - nothing is accepted automatically this time.
wiki_names = (wiki.dropna(subset=["capacity"])
                  .drop_duplicates("gkey").set_index("gkey"))
keys = list(wiki_names.index)

print("\n--- top 30 misses, with Wikipedia suggestions ---")
for _, r in (dim_venue[~hit]
             .sort_values("matches_played", ascending=False)
             .head(30).iterrows()):
    sugg = get_close_matches(r["gkey"], keys, n=2, cutoff=0.5)
    opts = " | ".join(
        f"{wiki_names.loc[s, 'Ground']} ({wiki_names.loc[s, 'Country']}, "
        f"{int(wiki_names.loc[s, 'capacity']):,})" for s in sugg
    ) or "no suggestion"
    print(f"{r['matches_played']:4d}  {r['venue_clean'][:42]:42s} "
          f"[{r['city_final'][:14]:14s}] -> {opts}")

venues with capacity : 105 of 236
matches covered      : 1,810 of 3,182 (57%)

--- top 30 misses, with Wikipedia suggestions ---
 128  Shere Bangla National Stadium              [Dhaka         ] -> Bangabandhu National Stadium (Bangladesh, 360,000) | Singapore National Stadium (Singapore, 52,000)
  48  Kennington Oval                            [London        ] -> Kensington Oval (Barbados, 11,000) | Karen Rolton Oval (Australia, 5,000)
  46  SuperSport Park                            [Centurion     ] -> Church Street Park (United States, 3,500) | Senwes Park (South Africa, 18,000)
  42  Seddon Park                                [Hamilton      ] -> Eden Park (New Zealand, 42,000) | Senwes Park (South Africa, 18,000)
  41  The Wanderers Stadium                      [Johannesburg  ] -> Wanderers Stadium (South Africa, 34,000) | Wankhede Stadium (India, 33,108)
  40  Sheikh Zayed Stadium                       [Abu Dhabi     ] -> Sheikh Zayed Cricket Stadium (United Arab Emirates, 20,000)

In [57]:
# Search the Wikipedia table directly for the grounds we're missing,
# so the numbers come from the source rather than from memory.
probe = ["gabba", "brisbane", "centurion", "supersport", "seddon",
         "basin", "bay oval", "george", "bindra", "mohali", "malahide",
         "stormont", "civil service", "sher-e", "sher e", "the oval",
         "waca", "perth", "amerat", "beausejour", "daren", "darren",
         "warner park", "maple leaf", "academy", "forthill",
         "windhoek", "hamilton", "mount maunganui"]

for p in probe:
    hits = wiki[wiki["Ground"].str.lower().str.contains(p, na=False)
                | wiki["City"].str.lower().str.contains(p, na=False)]
    print(f"\n--- '{p}' ---")
    if len(hits):
        print(hits[["Ground", "City", "Country", "capacity"]]
              .drop_duplicates().to_string(index=False))
    else:
        print("   not on the page")


--- 'gabba' ---
   Ground     City   Country  capacity
The Gabba Brisbane Australia   42000.0

--- 'brisbane' ---
                    Ground     City   Country  capacity
                 The Gabba Brisbane Australia   42000.0
        Allan Border Field Brisbane Australia    6500.0
Brisbane Exhibition Ground Brisbane Australia  254900.0

--- 'centurion' ---
        Ground      City      Country  capacity
Centurion Park Centurion South Africa   21000.0

--- 'supersport' ---
   not on the page

--- 'seddon' ---
   not on the page

--- 'basin' ---
   not on the page

--- 'bay oval' ---
   not on the page

--- 'george' ---
                         Ground           City      Country  capacity
                   Queen's Park Saint George's      Grenada   20000.0
St George's Park Cricket Ground Port Elizabeth South Africa   19000.0
 Georgetown Cricket Club Ground     Georgetown       Guyana   10000.0

--- 'bindra' ---
               Ground   City Country  capacity
PCA-IS Bindra Stadium Mohali

In [58]:
# The rename string-matching could never catch: Beausejour Stadium was
# renamed for St Lucia's World Cup-winning captain in 2016. Zero string
# similarity, same ground. Only domain knowledge finds this one.
matches["venue_clean"] = matches["venue_clean"].replace(
    {"Beausejour Stadium": "Daren Sammy National Cricket Stadium"})

dim_venue = (matches
             .groupby(["venue_clean", "city_final", "venue_country"], dropna=False)
             .agg(matches_played=("match_id", "size"))
             .reset_index()
             .sort_values(["venue_clean", "city_final"])
             .reset_index(drop=True))
print(f"dim_venue rows: {len(dim_venue)}   (was 236)")
print(f"total matches : {dim_venue['matches_played'].sum():,}  (must be 3,182)")

# --- home_nation ---
WEST_INDIES = {
    "Antigua and Barbuda", "Barbados", "Dominica", "Grenada", "Guyana",
    "Jamaica", "Saint Kitts and Nevis", "Saint Lucia",
    "Saint Vincent and the Grenadines", "Trinidad and Tobago",
}
dim_venue["home_nation"] = (dim_venue["venue_country"]
    .apply(lambda c: "West Indies" if c in WEST_INDIES else c)
    .replace({"The Netherlands": "Netherlands",
              "United States": "United States of America"}))
dim_venue.loc[dim_venue["home_nation"].isin({"Spain", "Malaysia"}),
              "home_nation"] = None

# --- capacity ---
WIKI_ALIASES = {
    "Shere Bangla National Stadium": "Sher-e-Bangla Cricket Stadium",
    "Kennington Oval": "The Oval",
    "SuperSport Park": "Centurion Park",
    "Brisbane Cricket Ground": "The Gabba",
    "St George's Park": "St George's Park Cricket Ground",
    "Civil Service Cricket Club": "Stormont",
    "The Village": "Malahide Cricket Club Ground",
    "Western Australia Cricket Association Ground": "WACA Ground",
    "Punjab Cricket Association IS Bindra Stadium": "PCA-IS Bindra Stadium",
    "Daren Sammy National Cricket Stadium": "Darren Sammy Cricket Ground",
    "Maple Leaf North-West Ground": "Maple Leaf Cricket Club",
    "The Wanderers Stadium": "Wanderers Stadium",
    "Sheikh Zayed Stadium": "Sheikh Zayed Cricket Stadium",
    "Kingsmead": "Kingsmead Cricket Ground",
    "Edgbaston": "Edgbaston Cricket Ground",
    "The Rose Bowl": "Rose Bowl",
    "Newlands": "Newlands Cricket Ground",
    "Zahur Ahmed Chowdhury Stadium": "Zohur Ahmed Chowdhury Stadium",
}

dim_venue["gkey"] = dim_venue["venue_clean"].map(squash)
dim_venue["ckey"] = dim_venue["city_final"].map(squash)
wiki_ok = wiki.dropna(subset=["capacity"])

# 1. name AND city agree
w1 = wiki_ok.drop_duplicates(["gkey", "ckey"]).set_index(["gkey", "ckey"])["capacity"]
cap1 = pd.Series(
    pd.MultiIndex.from_arrays([dim_venue["gkey"], dim_venue["ckey"]]).map(w1),
    index=dim_venue.index)

# 2. a name we checked by hand
by_ground = wiki_ok.drop_duplicates("Ground").set_index("Ground")["capacity"]
cap2 = dim_venue["venue_clean"].map(WIKI_ALIASES).map(by_ground)

# 3. name alone, only where unique on Wikipedia's side
counts = wiki["gkey"].value_counts()
w3 = (wiki_ok[wiki_ok["gkey"].isin(counts[counts == 1].index)]
      .set_index("gkey")["capacity"])
cap3 = dim_venue["gkey"].map(w3)

dim_venue["capacity"] = cap1.fillna(cap2).fillna(cap3)

# Record HOW each number was obtained - provenance, same principle
# as the 'how' column in your spreadsheet.
dim_venue["capacity_source"] = None
dim_venue.loc[cap3.notna(), "capacity_source"] = "wikipedia-name"
dim_venue.loc[cap2.notna(), "capacity_source"] = "wikipedia-verified"
dim_venue.loc[cap1.notna(), "capacity_source"] = "wikipedia-exact"

hit = dim_venue["capacity"].notna()
total = dim_venue["matches_played"].sum()
print(f"\nvenues with capacity : {hit.sum()} of {len(dim_venue)}")
print(f"matches covered      : {dim_venue.loc[hit, 'matches_played'].sum():,} "
      f"of {total:,} ({dim_venue.loc[hit, 'matches_played'].sum() / total:.0%})")
print("\nby source:")
print(dim_venue["capacity_source"].value_counts(dropna=False).to_string())

# --- keys and display name ---
dim_venue.insert(0, "venue_id", range(1, len(dim_venue) + 1))
ambiguous = set(dim_venue["venue_clean"].value_counts().loc[lambda s: s > 1].index)
dim_venue["venue_display"] = [
    f"{v}, {c}" if v in ambiguous else v
    for v, c in zip(dim_venue["venue_clean"], dim_venue["city_final"])
]

lookup = dim_venue.set_index(["venue_clean", "city_final"])["venue_id"]
matches["venue_id"] = pd.MultiIndex.from_arrays(
    [matches["venue_clean"], matches["city_final"]]).map(lookup)
print(f"\nmatches with no venue_id: {matches['venue_id'].isna().sum()}")

out = dim_venue[["venue_id", "venue_clean", "venue_display", "city_final",
                 "venue_country", "home_nation", "capacity",
                 "capacity_source", "matches_played"]].rename(
    columns={"venue_clean": "venue_name", "city_final": "city",
             "venue_country": "country"})
out.to_parquet(PROCESSED / "dim_venue.parquet", index=False)
print(f"saved -> dim_venue.parquet  ({len(out)} rows)")

print("\n--- top 15 ---")
print(out.sort_values("matches_played", ascending=False)
      .head(15).to_string(index=False))

dim_venue rows: 235   (was 236)
total matches : 3,182  (must be 3,182)

venues with capacity : 123 of 235
matches covered      : 2,442 of 3,182 (77%)

by source:
capacity_source
None                  112
wikipedia-exact        89
wikipedia-verified     18
wikipedia-name         16

matches with no venue_id: 0
saved -> dim_venue.parquet  (235 rows)

--- top 15 ---
 venue_id                                        venue_name                                     venue_display      city              country          home_nation  capacity    capacity_source  matches_played
      187                     Shere Bangla National Stadium                     Shere Bangla National Stadium     Dhaka           Bangladesh           Bangladesh   25416.0 wikipedia-verified             128
       78                                Harare Sports Club                                Harare Sports Club    Harare             Zimbabwe             Zimbabwe   10000.0    wikipedia-exact             127
      161    

In [59]:
# Keyed on (team_name, gender), because "England" is two different
# international sides and merging them would blend two competitions.
long = pd.concat([
    matches[["team1", "gender"]].rename(columns={"team1": "team_name"}),
    matches[["team2", "gender"]].rename(columns={"team2": "team_name"}),
])

dim_team = (long.dropna(subset=["team_name"])
            .groupby(["team_name", "gender"]).size()
            .reset_index(name="matches_played")
            .sort_values(["team_name", "gender"])
            .reset_index(drop=True))

dim_team.insert(0, "team_id", range(1, len(dim_team) + 1))
dim_team["team_display"] = [
    n if g == "male" else f"{n} Women"
    for n, g in zip(dim_team["team_name"], dim_team["gender"])
]

print(f"dim_team rows: {len(dim_team)}")
print(f"teams fielding both a men's and women's side: "
      f"{(dim_team.groupby('team_name').size() > 1).sum()}")

# Attach the key to all four team columns on the match.
lookup = dim_team.set_index(["team_name", "gender"])["team_id"]
for col in ["team1", "team2", "toss_winner", "winner"]:
    matches[f"{col}_id"] = pd.MultiIndex.from_arrays(
        [matches[col], matches["gender"]]).map(lookup)
    missing = matches[col].notna() & matches[f"{col}_id"].isna()
    print(f"  {col:12s} unresolved: {missing.sum()}")

dim_team.to_parquet(PROCESSED / "dim_team.parquet", index=False)
print(f"\nsaved -> dim_team.parquet")
print(dim_team.to_string(index=False))

dim_team rows: 44
teams fielding both a men's and women's side: 16
  team1        unresolved: 0
  team2        unresolved: 0
  toss_winner  unresolved: 0
  winner       unresolved: 0

saved -> dim_team.parquet
 team_id                team_name gender  matches_played                   team_display
       1                Africa XI   male               5                      Africa XI
       2                  Asia XI   male               5                        Asia XI
       3                Australia female             125                Australia Women
       4                Australia   male             476                      Australia
       5               Bangladesh female              62               Bangladesh Women
       6               Bangladesh   male             333                     Bangladesh
       7                  Bermuda   male              12                        Bermuda
       8                   Canada   male              68                         Canad

In [60]:
matches["match_date"] = pd.to_datetime(matches["match_date"])

print(f"date range : {matches['match_date'].min().date()} "
      f"-> {matches['match_date'].max().date()}")
print(f"null dates : {matches['match_date'].isna().sum()}")

print("\nmatches per year:")
print(matches["match_date"].dt.year.value_counts().sort_index().to_string())

# --- dim_date ---
# A row for EVERY calendar day in the range, not just days with a
# match. That's what makes "matches per quarter" able to show a
# quarter with zero matches instead of skipping it.
rng = pd.date_range(matches["match_date"].min().normalize(),
                    matches["match_date"].max().normalize(), freq="D")

dim_date = pd.DataFrame({"full_date": rng})
dim_date.insert(0, "date_key",
                dim_date["full_date"].dt.strftime("%Y%m%d").astype(int))
dim_date["year"] = dim_date["full_date"].dt.year
dim_date["quarter"] = dim_date["full_date"].dt.quarter
dim_date["month"] = dim_date["full_date"].dt.month
dim_date["month_name"] = dim_date["full_date"].dt.month_name()
dim_date["day"] = dim_date["full_date"].dt.day
dim_date["day_name"] = dim_date["full_date"].dt.day_name()
dim_date["is_weekend"] = dim_date["full_date"].dt.dayofweek >= 5

matches["date_key"] = (matches["match_date"].dt.strftime("%Y%m%d")
                       .astype("Int64"))

print(f"\ndim_date rows: {len(dim_date):,}")
print(f"matches with no date_key: {matches['date_key'].isna().sum()}")
print(dim_date.head(3).to_string(index=False))

# --- what does the series data look like? ---
print(f"\n\nseries_name nulls   : {matches['series_name'].isna().sum()}")
print(f"distinct series     : {matches['series_name'].nunique()}")
print("\nbiggest series:")
print(matches["series_name"].value_counts().head(15).to_string())

date range : 2002-06-27 -> 2026-09-09
null dates : 0

matches per year:
match_date
2002      3
2003    119
2004     79
2005     67
2006    119
2007    154
2008    104
2009    119
2010    105
2011    140
2012     91
2013    148
2014    118
2015    138
2016    112
2017    169
2018    136
2019    167
2020     50
2021    110
2022    225
2023    245
2024    159
2025    202
2026    103

dim_date rows: 8,841
matches with no date_key: 0
 date_key  full_date  year  quarter  month month_name  day day_name  is_weekend
 20020627 2002-06-27  2002        2      6       June   27 Thursday       False
 20020628 2002-06-28  2002        2      6       June   28   Friday       False
 20020629 2002-06-29  2002        2      6       June   29 Saturday        True


series_name nulls   : 14
distinct series     : 398

biggest series:
series_name
ICC Men's Cricket World Cup League 2    234
ICC Cricket World Cup                   130
ICC Women's World Cup                   102
ICC World Cup                    

In [61]:
print(f"distinct series names        : {matches['series_name'].nunique()}")
print(f"distinct (series, season)    : "
      f"{matches.groupby(['series_name', 'season'], dropna=False).ngroups}")

# How are the World Cups actually spelled, and across which seasons?
wc = matches[matches["series_name"].str.contains("world cup", case=False, na=False)]
probe = (wc.groupby(["series_name", "gender"])
           .agg(matches=("match_id", "size"),
                seasons=("season", lambda s: ", ".join(sorted(set(s.astype(str))))))
           .reset_index()
           .sort_values("matches", ascending=False))
print("\n--- everything with 'World Cup' in the name ---")
print(probe.to_string(index=False))

# The 14 matches with no series at all - what are they?
print("\n--- matches with no series_name ---")
print(matches[matches["series_name"].isna()]
      [["match_id", "match_date", "season", "gender", "team1", "team2"]]
      .sort_values("match_date").to_string(index=False))

distinct series names        : 398
distinct (series, season)    : 716

--- everything with 'World Cup' in the name ---
                                 series_name gender  matches                                                                                  seasons
        ICC Men's Cricket World Cup League 2   male      234 2019, 2019/20, 2021, 2021/22, 2022, 2022/23, 2023/24, 2024, 2024/25, 2025, 2025/26, 2026
                       ICC Cricket World Cup   male      130                                                                2010/11, 2014/15, 2023/24
                       ICC Women's World Cup female      102                                                 2008/09, 2012/13, 2017, 2021/22, 2025/26
                               ICC World Cup   male       99                                                                         2002/03, 2006/07
             ICC Cricket World Cup Qualifier   male       38                                                                       

In [62]:
SERIES_ALIASES = {
    "ICC World Cup": "ICC Cricket World Cup",
    "World Cup": "ICC Cricket World Cup",
    "ICC World Cup Qualifiers": "ICC Cricket World Cup Qualifier",
    "ICC Cricket World Cup Qualifier (ICC Trophy)":
        "ICC Cricket World Cup Qualifier",
    "ICC Women's Cricket World Cup Qualifier": "ICC Women's World Cup Qualifier",
    "ICC Women's World Cup Qualifying Series": "ICC Women's World Cup Qualifier",
}

matches["series_clean"] = matches["series_name"].replace(SERIES_ALIASES)

# "2010/11" -> 2010, "2019" -> 2019. Gives us something sortable.
matches["season_start"] = (matches["season"].astype(str)
                           .str.slice(0, 4).astype(int))

named = matches[matches["series_name"].notna()]

dim_series = (named.groupby(["series_clean", "season", "gender"])
              .agg(matches_played=("match_id", "size"),
                   season_start=("season_start", "first"),
                   first_match=("match_date", "min"),
                   last_match=("match_date", "max"))
              .reset_index()
              .sort_values(["season_start", "series_clean"])
              .reset_index(drop=True))
dim_series.insert(0, "series_id", range(1, len(dim_series) + 1))

# The Unknown member. Keeps the foreign key non-null so an INNER JOIN
# can't silently swallow these matches.
unknown = pd.DataFrame([{
    "series_id": 0, "series_clean": "No series recorded",
    "season": None, "gender": None,
    "matches_played": int(matches["series_name"].isna().sum()),
    "season_start": None, "first_match": None, "last_match": None,
}])
dim_series = pd.concat([unknown, dim_series], ignore_index=True)

lookup = dim_series.set_index(["series_clean", "season", "gender"])["series_id"]
matches["series_id"] = pd.MultiIndex.from_arrays(
    [matches["series_clean"], matches["season"], matches["gender"]]).map(lookup)
matches["series_id"] = matches["series_id"].fillna(0).astype(int)

print(f"dim_series rows          : {len(dim_series)}")
print(f"matches on series_id = 0 : {(matches['series_id'] == 0).sum()}  (should be 14)")
print(f"matches with no series_id: {matches['series_id'].isna().sum()}")

print("\n--- the six men's World Cups, now one series ---")
print(dim_series[dim_series["series_clean"] == "ICC Cricket World Cup"]
      [["series_id", "season", "gender", "matches_played",
        "first_match", "last_match"]].to_string(index=False))

dim_series.to_parquet(PROCESSED / "dim_series.parquet", index=False)
dim_date.to_parquet(PROCESSED / "dim_date.parquet", index=False)
print("\nsaved -> dim_series.parquet, dim_date.parquet")

dim_series rows          : 707
matches on series_id = 0 : 14  (should be 14)
matches with no series_id: 0

--- the six men's World Cups, now one series ---
 series_id  season gender  matches_played         first_match          last_match
         2 2002/03   male              49 2003-02-09 00:00:00 2003-03-23 00:00:00
        80 2006/07   male              50 2007-03-13 00:00:00 2007-04-28 00:00:00
       181 2010/11   male              49 2011-02-19 00:00:00 2011-04-02 00:00:00
       281 2014/15   male              42 2015-02-14 00:00:00 2015-03-29 00:00:00
       438    2019   male              36 2019-05-30 00:00:00 2019-07-14 00:00:00
       576 2023/24   male              39 2023-10-05 00:00:00 2023-11-19 00:00:00

saved -> dim_series.parquet, dim_date.parquet


In [63]:
# Everything played inside the 2019 World Cup window, by series name.
wc19 = matches[(matches["match_date"] >= "2019-05-30")
               & (matches["match_date"] <= "2019-07-14")
               & (matches["gender"] == "male")]

print(f"male ODIs in the 2019 World Cup window: {len(wc19)}")
print(wc19["series_clean"].value_counts(dropna=False).to_string())

print("\n--- the ones NOT labelled World Cup ---")
other = wc19[wc19["series_clean"] != "ICC Cricket World Cup"]
print(other[["match_date", "series_clean", "team1", "team2", "venue_clean"]]
      .sort_values("match_date").to_string(index=False))

male ODIs in the 2019 World Cup window: 41
series_clean
ICC Cricket World Cup                       36
Zimbabwe tour of Netherlands and Ireland     5

--- the ones NOT labelled World Cup ---
match_date                             series_clean       team1    team2                venue_clean
2019-06-19 Zimbabwe tour of Netherlands and Ireland Netherlands Zimbabwe  Sportpark Het Schootsveld
2019-06-21 Zimbabwe tour of Netherlands and Ireland Netherlands Zimbabwe  Sportpark Het Schootsveld
2019-07-01 Zimbabwe tour of Netherlands and Ireland     Ireland Zimbabwe        Bready Cricket Club
2019-07-04 Zimbabwe tour of Netherlands and Ireland     Ireland Zimbabwe Civil Service Cricket Club
2019-07-07 Zimbabwe tour of Netherlands and Ireland     Ireland Zimbabwe Civil Service Cricket Club


In [64]:
import zipfile, json

RAW = root / "data" / "raw"
zpath = RAW / "odis_json.zip"

in_window, sample = 0, []
with zipfile.ZipFile(zpath) as z:
    for name in z.namelist():
        if not name.endswith(".json"):
            continue
        with z.open(name) as f:
            info = json.load(f)["info"]
        dates = info.get("dates") or []
        if not dates:
            continue
        if info.get("gender") != "male":
            continue
        if "2019-05-30" <= str(dates[0]) <= "2019-07-14":
            in_window += 1
            event = info.get("event")
            ename = event.get("name") if isinstance(event, dict) else event
            sample.append((name, str(dates[0]), ename,
                           " v ".join(info.get("teams", []))))

print(f"male ODIs in the zip, 2019 WC window : {in_window}")
print(f"male ODIs in matches.parquet, same window : 41")

print("\n--- what the zip actually contains ---")
for n, d, e, t in sorted(sample, key=lambda x: x[1]):
    print(f"{d}  {str(e)[:35]:35s}  {t}")

male ODIs in the zip, 2019 WC window : 41
male ODIs in matches.parquet, same window : 41

--- what the zip actually contains ---
2019-05-30  World Cup                            England v South Africa
2019-05-31  World Cup                            Pakistan v West Indies
2019-06-01  World Cup                            New Zealand v Sri Lanka
2019-06-02  World Cup                            Bangladesh v South Africa
2019-06-03  World Cup                            England v Pakistan
2019-06-05  World Cup                            India v South Africa
2019-06-05  World Cup                            Bangladesh v New Zealand
2019-06-06  World Cup                            Australia v West Indies
2019-06-08  World Cup                            England v Bangladesh
2019-06-09  World Cup                            Australia v India
2019-06-10  World Cup                            South Africa v West Indies
2019-06-12  World Cup                            Australia v Pakistan
2019-06-14 

In [65]:
from collections import Counter

team_counts = Counter()
genders = Counter()
total = 0

with zipfile.ZipFile(zpath) as z:
    for name in z.namelist():
        if not name.endswith(".json"):
            continue
        with z.open(name) as f:
            info = json.load(f)["info"]
        total += 1
        genders[info.get("gender")] += 1
        for t in info.get("teams", []):
            team_counts[t] += 1

print(f"json files in the zip : {total}")
print(f"by gender             : {dict(genders)}")
print(f"distinct teams        : {len(team_counts)}")

print("\n--- every team in the zip, by match count ---")
for t, c in sorted(team_counts.items(), key=lambda x: -x[1]):
    print(f"  {t:32s} {c}")

print(f"\nAfghanistan matches in the zip: {team_counts.get('Afghanistan', 0)}")

json files in the zip : 3182
by gender             : {'male': 2571, 'female': 611}
distinct teams        : 28

--- every team in the zip, by match count ---
  India                            654
  Australia                        601
  England                          589
  Sri Lanka                        582
  South Africa                     543
  Pakistan                         534
  New Zealand                      517
  West Indies                      498
  Bangladesh                       395
  Zimbabwe                         314
  Ireland                          202
  Scotland                         135
  United Arab Emirates             124
  Netherlands                      112
  Nepal                            88
  United States of America         87
  Namibia                          76
  Papua New Guinea                 71
  Oman                             71
  Canada                           68
  Kenya                            41
  Hong Kong                    

In [66]:
for f in ["squads", "people", "batting", "bowling"]:
    df = pd.read_parquet(PROCESSED / f"{f}.parquet")
    print(f"--- {f}.parquet  ({len(df):,} rows) ---")
    print(f"columns: {list(df.columns)}")
    print(df.head(3).to_string(index=False))
    print()

--- squads.parquet  (70,121 rows) ---
columns: ['match_id', 'team', 'player_id', 'player_name']
 match_id      team player_id player_name
  1000887 Australia  dcce6f09   DA Warner
  1000887 Australia  12b610c2     TM Head
  1000887 Australia  30a45b23   SPD Smith

--- people.parquet  (3,262 rows) ---
columns: ['cricsheet_id', 'player_name', 'aliases', 'name_count']
cricsheet_id  player_name aliases  name_count
    004c9e85    PJ Hughes     NaN           1
    005f0561  Panna Ghosh     NaN           1
    006de9ca Zulfiqar Jan     NaN           1

--- batting.parquet  (56,052 rows) ---
columns: ['player', 'team', 'innings_no', 'batting_position', 'runs_scored', 'balls_faced', 'fours', 'sixes', 'dismissal_kind', 'dismissed_by', 'fielder', 'is_not_out', 'match_id', 'player_id', 'dismissed_by_id', 'fielder_id']
   player      team  innings_no  batting_position  runs_scored  balls_faced  fours  sixes dismissal_kind  dismissed_by         fielder  is_not_out  match_id player_id dismissed_by_i

In [67]:
squads = pd.read_parquet(PROCESSED / "squads.parquet")
people = pd.read_parquet(PROCESSED / "people.parquet")
batting = pd.read_parquet(PROCESSED / "batting.parquet")
bowling = pd.read_parquet(PROCESSED / "bowling.parquet")

# Team sheets define who is a player - NOT the registry, which also
# contains umpires and match referees.
base = (squads.groupby("player_id")
        .agg(matches_played=("match_id", "nunique"))
        .reset_index())

name_map = people.set_index("cricsheet_id")["player_name"]
fallback = squads.groupby("player_id")["player_name"].agg(
    lambda s: s.value_counts().index[0])
base["player_name"] = (base["player_id"].map(name_map)
                       .fillna(base["player_id"].map(fallback)))

squads["gender"] = squads["match_id"].map(matches.set_index("match_id")["gender"])
squads["match_date"] = squads["match_id"].map(
    matches.set_index("match_id")["match_date"])

agg = (squads.groupby("player_id")
       .agg(gender=("gender", lambda s: s.mode().iloc[0]),
            primary_team=("team", lambda s: s.value_counts().index[0]),
            teams_played_for=("team", "nunique"),
            debut=("match_date", "min"),
            last_match=("match_date", "max"))
       .reset_index())
base = base.merge(agg, on="player_id", how="left")

# --- the three signals the role rule uses ---
bowl_m = bowling.groupby("player_id")["match_id"].nunique().rename("matches_bowled")
bat_m = batting.groupby("player_id")["match_id"].nunique().rename("matches_batted")
pos = batting.groupby("player_id")["batting_position"].mean().rename("avg_position")
catches = (batting[batting["dismissal_kind"].isin(["caught", "stumped"])]
           .groupby("fielder_id").size().rename("catches_stumpings"))

for s in (bowl_m, bat_m, pos, catches):
    base = base.merge(s, left_on="player_id", right_index=True, how="left")

base[["matches_bowled", "matches_batted", "catches_stumpings"]] = \
    base[["matches_bowled", "matches_batted", "catches_stumpings"]].fillna(0)

base["bowl_share"] = base["matches_bowled"] / base["matches_played"]
base["dismissal_rate"] = base["catches_stumpings"] / base["matches_played"]


def classify_role(row) -> str:
    """Order matters. Bowling is checked first because a keeper cannot
    bowl while keeping - a high bowl_share means the gloves were
    incidental. Keepers are found by dismissal RATE, not stumpings:
    stumpings only happen off spin, so a stumping count measures how
    much spin a team bowled, not who kept wicket."""
    if row["matches_batted"] == 0 and row["matches_bowled"] == 0:
        return "Unknown"
    if row["bowl_share"] >= 0.40:
        return "All-rounder" if row["avg_position"] <= 7 else "Bowler"
    if row["dismissal_rate"] >= 0.70:
        return "Wicket-keeper"
    return "Batsman"


base["playing_role"] = base.apply(classify_role, axis=1)

# Deferred to Day 12 - Wikipedia/Wikidata enrichment.
base["batting_style"] = None
base["bowling_style"] = None

dim_player = base.sort_values("player_name").reset_index(drop=True)

print(f"dim_player rows: {len(dim_player)}   (expected 2,694)")
print("\nrole distribution:")
print(dim_player["playing_role"].value_counts().to_string())
print("\ngender split:")
print(dim_player["gender"].value_counts().to_string())

# Every player in the facts must exist in the dimension.
for nm, df in [("batting", batting), ("bowling", bowling)]:
    orphans = set(df["player_id"].dropna()) - set(dim_player["player_id"])
    print(f"\n{nm}: player_ids not in dim_player -> {len(orphans)}")

dim_player.to_parquet(PROCESSED / "dim_player.parquet", index=False)
print(f"\nsaved -> dim_player.parquet")

print("\n--- most-capped players ---")
print(dim_player.sort_values("matches_played", ascending=False)
      [["player_name", "primary_team", "gender", "playing_role",
        "matches_played", "debut", "last_match"]]
      .head(12).to_string(index=False))

dim_player rows: 2694   (expected 2,694)

role distribution:
playing_role
Bowler           1224
Batsman           776
All-rounder       474
Wicket-keeper     207
Unknown            13

gender split:
gender
male      1994
female     700

batting: player_ids not in dim_player -> 0

bowling: player_ids not in dim_player -> 0

saved -> dim_player.parquet

--- most-capped players ---
     player_name primary_team gender  playing_role  matches_played      debut last_match
        MS Dhoni        India   male Wicket-keeper             331 2004-12-23 2019-07-09
         V Kohli        India   male       Batsman             311 2008-08-18 2026-07-19
   KC Sangakkara    Sri Lanka   male Wicket-keeper             298 2002-06-27 2015-03-18
DPMD Jayawardene    Sri Lanka   male       Batsman             283 2002-06-27 2015-03-18
       RG Sharma        India   male       Batsman             281 2007-06-23 2026-07-19
      TM Dilshan    Sri Lanka   male   All-rounder             280 2003-05-19 2016-0

In [68]:
REF = root / "etl" / "reference"
REF.mkdir(parents=True, exist_ok=True)

# 1. Your manual venue corrections - human work, not reproducible.
manual = pd.read_csv(PROCESSED / "venues_manual.csv")
manual.to_csv(REF / "venues_manual.csv", index=False)
print(f"saved {len(manual)} rows -> etl/reference/venues_manual.csv")

# 2. The Wikipedia capacities, frozen. The ETL reads this file, not
#    the live page, so re-running it a year from now gives the same
#    answer. Delete the file to deliberately refresh.
cap = (dim_venue[dim_venue["capacity"].notna()]
       [["venue_clean", "city_final", "capacity", "capacity_source"]]
       .rename(columns={"venue_clean": "venue_name", "city_final": "city"}))
cap["capacity"] = cap["capacity"].astype(int)
cap["retrieved"] = pd.Timestamp.today().date().isoformat()
cap.to_csv(REF / "venue_capacity.csv", index=False)
print(f"saved {len(cap)} rows -> etl/reference/venue_capacity.csv")

print("\nfiles now in etl/reference/:")
for f in sorted(REF.iterdir()):
    print(f"   {f.name:28s} {f.stat().st_size / 1024:6.1f} KB")

saved 382 rows -> etl/reference/venues_manual.csv
saved 123 rows -> etl/reference/venue_capacity.csv

files now in etl/reference/:
   venue_capacity.csv              7.8 KB
   venues_manual.csv              24.2 KB


In [70]:
# Use whichever column names this DataFrame happens to have.
cols = dim_venue.columns
name_col = "venue_display" if "venue_display" in cols else "venue_clean"
city_col = "city" if "city" in cols else "city_final"
ctry_col = "country" if "country" in cols else "venue_country"

missing = (dim_venue[dim_venue["capacity"].isna()]
           .sort_values("matches_played", ascending=False))

print(f"{len(missing)} grounds with no capacity "
      f"({missing['matches_played'].sum():,} matches)\n")

for i, (_, r) in enumerate(missing.iterrows(), 1):
    print(f"{i:3d}. {str(r[name_col]):<52s} {str(r[city_col]):<18s} "
          f"{str(r[ctry_col]):<22s} {r['matches_played']:>4d}")

112 grounds with no capacity (740 matches)

  1. Seddon Park                                          Hamilton           New Zealand              42
  2. Al Amerat Cricket Ground Oman Cricket (Ministry Turf 1) Al Amarat          Oman                     28
  3. ICC Academy                                          Dubai              United Arab Emirates     28
  4. Bay Oval                                             Mount Maunganui    New Zealand              28
  5. Warner Park                                          Basseterre         Saint Kitts and Nevis    27
  6. County Ground, Bristol                               Bristol            England                  26
  7. VRA Ground                                           Amstelveen         The Netherlands          22
  8. Westpac Stadium                                      Wellington         New Zealand              22
  9. Wanderers Cricket Ground                             Windhoek           Namibia                  21
 10. Bas

In [71]:
dv = pd.read_parquet(PROCESSED / "dim_venue.parquet")

def venue_card(r) -> str:
    """Exactly what the Streamlit venue panel will show."""
    cap = f" · capacity {int(r['capacity']):,}" if pd.notna(r["capacity"]) else ""
    lines = [r["venue_name"], f"{r['city']}, {r['country']}{cap}"]
    if pd.notna(r["former_names"]):
        lines.append(f"formerly {r['former_names']}")
    lines += ["", f"Matches hosted   {r['matches_played']}"]
    return "\n".join(lines)

for v in ["Arun Jaitley Stadium", "Mangaung Oval", "Narendra Modi Stadium",
          "R Premadasa Stadium", "Seddon Park"]:
    row = dv[dv["venue_name"] == v]
    if len(row):
        print(venue_card(row.iloc[0]))
        print("-" * 46)

print("\n--- every ground carrying a former name ---")
print(dv[dv["former_names"].notna()]
      [["venue_display", "city", "capacity", "former_names", "matches_played"]]
      .sort_values("matches_played", ascending=False)
      .to_string(index=False))

Arun Jaitley Stadium
Delhi, India · capacity 35,200
formerly Feroz Shah Kotla

Matches hosted   18
----------------------------------------------
Mangaung Oval
Bloemfontein, South Africa · capacity 20,000
formerly Chevrolet Park, Goodyear Park, OUTsurance Oval

Matches hosted   20
----------------------------------------------
Narendra Modi Stadium
Ahmedabad, India · capacity 132,000
formerly Sardar Patel Stadium

Matches hosted   22
----------------------------------------------
R Premadasa Stadium
Colombo, Sri Lanka · capacity 35,000
formerly Khettarama Stadium

Matches hosted   108
----------------------------------------------
Seddon Park
Hamilton, New Zealand
formerly Westpac Park

Matches hosted   43
----------------------------------------------

--- every ground carrying a former name ---
                                               venue_display          city  capacity                                                                former_names  matches_played
               

In [72]:
dv = pd.read_parquet(PROCESSED / "dim_venue.parquet")

print(f"venues: {len(dv)}")
print(f"max capacity: {dv['capacity'].max():,.0f}  (must be <= 132,000)")
print(f"with capacity: {dv['capacity'].notna().sum()}")

# Bug 1: the two-step rename should now be followed all the way.
print("\nany row still called 'Sardar Patel Stadium':",
      (dv["venue_name"] == "Sardar Patel Stadium").sum(), "(want 0)")

# Bug 3: each County Ground should list only its OWN former names.
print("\n--- County Grounds ---")
print(dv[dv["venue_name"] == "County Ground"]
      [["venue_display", "city", "capacity", "former_names", "matches_played"]]
      .to_string(index=False))

print("\n--- Ahmedabad ---")
print(dv[dv["city"] == "Ahmedabad"]
      [["venue_name", "capacity", "former_names", "matches_played"]]
      .to_string(index=False))

venues: 226
max capacity: 132,000  (must be <= 132,000)
with capacity: 115

any row still called 'Sardar Patel Stadium': 0 (want 0)

--- County Grounds ---
             venue_display        city  capacity                           former_names  matches_played
    County Ground, Bristol     Bristol       NaN The Royal & Sun Alliance County Ground              26
 County Ground, Chelmsford  Chelmsford       NaN                                    NaN               6
      County Ground, Derby       Derby       NaN                                    NaN              12
       County Ground, Hove        Hove       NaN                                    NaN               4
County Ground, Northampton Northampton    6500.0                                    NaN               2
    County Ground, Taunton     Taunton   12500.0    The Cooper Associates County Ground              17
  County Ground, Worcester   Worcester       NaN                                    NaN               8

--- Ahmedab

In [73]:
dv = pd.read_parquet(PROCESSED / "dim_venue.parquet")
no_cap = dv[dv["capacity"].isna()].sort_values("matches_played", ascending=False)

print(f"{len(no_cap)} venues with no capacity\n")
for n in no_cap["venue_display"]:
    print(n)

# and as a plain text file you can open or paste elsewhere
out_txt = PROCESSED / "venues_no_capacity.txt"
out_txt.write_text("\n".join(no_cap["venue_display"]), encoding="utf-8")
print(f"\nsaved -> {out_txt}")

111 venues with no capacity

Seddon Park
Bay Oval
Al Amerat Cricket Ground Oman Cricket (Ministry Turf 1)
ICC Academy
Warner Park
County Ground, Bristol
Westpac Stadium
VRA Ground
Wanderers Cricket Ground
Basin Reserve
United Cricket Club Ground
Forthill
Saxton Oval
Sinhalese Sports Club Ground
Sportpark Maarschalkerweerd
Diamond Oval
Clontarf Cricket Club Ground
Moosa Cricket Stadium
County Ground, Derby
Arnos Vale Ground
Central Broward Regional Park Stadium Turf Ground
Castle Avenue
Rajiv Gandhi International Stadium
Reliance Stadium
Grange Cricket Club Ground
Takashinga Sports Club
Jade Stadium
Himachal Pradesh Cricket Association Stadium
John Davies Oval
County Ground, Worcester
Hazelaarweg
Lahore City Cricket Association Ground
Coolidge Cricket Ground
Barsapara Cricket Stadium
Bert Sutcliffe Oval
Saurashtra Cricket Association Stadium
JSCA International Stadium Complex
Bulawayo Athletic Club
Green Park
Cazaly's Stadium
County Ground, Chelmsford
Dr DY Patil Sports Academy
Al Amera

In [74]:
dv = pd.read_parquet(PROCESSED / "dim_venue.parquet")
hit = dv["capacity"].notna()
total = dv["matches_played"].sum()

print(f"venues with capacity : {hit.sum()} of {len(dv)}")
print(f"matches covered      : {dv.loc[hit,'matches_played'].sum():,} of {total:,} "
      f"({dv.loc[hit,'matches_played'].sum()/total:.0%})")
print(f"\nby source:")
print(dv["capacity_source"].value_counts(dropna=False).to_string())

print(f"\nstill missing ({(~hit).sum()}), by matches:")
print(dv[~hit].sort_values("matches_played", ascending=False)
      [["venue_display", "city", "country", "matches_played"]]
      .head(15).to_string(index=False))

venues with capacity : 197 of 226
matches covered      : 3,103 of 3,182 (98%)

by source:
capacity_source
wikipedia-exact       84
manual                82
NaN                   29
wikipedia-verified    18
wikipedia-name        13

still missing (29), by matches:
                             venue_display          city      country  matches_played
                  Wanderers Cricket Ground      Windhoek      Namibia              21
                    Takashinga Sports Club        Harare     Zimbabwe               9
    Lahore City Cricket Association Ground        Lahore     Pakistan               7
                             Old Hararians        Harare     Zimbabwe               5
                 Royal Chiangmai Golf Club    Chiang Mai     Thailand               4
          North-West University No1 Ground Potchefstroom South Africa               3
                     Cambusdoon New Ground           Ayr     Scotland               3
Sheikh Kamal International Cricket Stadium   Cox

In [75]:
mk = pd.read_parquet(PROCESSED / "matches_keyed.parquet")
dv = pd.read_parquet(PROCESSED / "dim_venue.parquet")
dt = pd.read_parquet(PROCESSED / "dim_team.parquet")
ds = pd.read_parquet(PROCESSED / "dim_series.parquet")
dd = pd.read_parquet(PROCESSED / "dim_date.parquet")
dp = pd.read_parquet(PROCESSED / "dim_player.parquet")
bat = pd.read_parquet(PROCESSED / "batting.parquet")
bwl = pd.read_parquet(PROCESSED / "bowling.parquet")

def fk(label, child_values, parent_values):
    """A foreign key is valid when every non-null child value exists
    in the parent. This is the check the database will enforce on
    Day 6 - better to fail it here, where the error is readable."""
    child = set(pd.Series(child_values).dropna())
    missing = child - set(parent_values)
    flag = "PASS" if not missing else "FAIL"
    extra = "" if not missing else f"  e.g. {sorted(missing)[:3]}"
    print(f"  [{flag}] {label:<42s} {len(missing)} broken{extra}")

print("referential integrity\n")
fk("matches.venue_id -> dim_venue", mk["venue_id"], dv["venue_id"])
fk("matches.series_id -> dim_series", mk["series_id"], ds["series_id"])
fk("matches.date_key -> dim_date", mk["date_key"], dd["date_key"])
for c in ["team1_id", "team2_id", "toss_winner_id", "winner_id"]:
    fk(f"matches.{c} -> dim_team", mk[c], dt["team_id"])

fk("batting.match_id -> matches", bat["match_id"], mk["match_id"])
fk("bowling.match_id -> matches", bwl["match_id"], mk["match_id"])
fk("batting.player_id -> dim_player", bat["player_id"], dp["player_id"])
fk("batting.dismissed_by_id -> dim_player", bat["dismissed_by_id"], dp["player_id"])
fk("batting.fielder_id -> dim_player", bat["fielder_id"], dp["player_id"])
fk("bowling.player_id -> dim_player", bwl["player_id"], dp["player_id"])

print(f"\nrow counts")
for n, d in [("matches", mk), ("batting", bat), ("bowling", bwl),
             ("dim_venue", dv), ("dim_team", dt), ("dim_series", ds),
             ("dim_date", dd), ("dim_player", dp)]:
    print(f"  {n:<12s} {len(d):>7,}")

# --- the venue list ---
vl = (dv[["venue_id", "venue_display", "city", "country",
          "capacity", "capacity_source", "matches_played"]]
      .sort_values("capacity", ascending=False, na_position="last"))

vl.to_csv(PROCESSED / "venue_list.csv", index=False)
print(f"\nsaved -> data/processed/venue_list.csv  ({len(vl)} rows)\n")
print(vl.to_string(index=False))

referential integrity

  [PASS] matches.venue_id -> dim_venue              0 broken
  [PASS] matches.series_id -> dim_series            0 broken
  [PASS] matches.date_key -> dim_date               0 broken
  [PASS] matches.team1_id -> dim_team               0 broken
  [PASS] matches.team2_id -> dim_team               0 broken
  [PASS] matches.toss_winner_id -> dim_team         0 broken
  [PASS] matches.winner_id -> dim_team              0 broken
  [PASS] batting.match_id -> matches                0 broken
  [PASS] bowling.match_id -> matches                0 broken
  [PASS] batting.player_id -> dim_player            0 broken
  [PASS] batting.dismissed_by_id -> dim_player      0 broken
  [FAIL] batting.fielder_id -> dim_player           17 broken  e.g. ['04985f66', '10d94c90', '149d0c18']
  [PASS] bowling.player_id -> dim_player            0 broken

row counts
  matches        3,182
  batting       56,052
  bowling       38,403
  dim_venue        226
  dim_team          44
  dim_series 

In [76]:
orphans = sorted(set(bat["fielder_id"].dropna()) - set(dp["player_id"]))
print(f"orphan fielder ids: {len(orphans)}\n")

people = pd.read_parquet(PROCESSED / "people.parquet")
known = people[people["cricsheet_id"].isin(orphans)]
print(f"how many are in the registry: {len(known)} of {len(orphans)}")
print(known[["cricsheet_id", "player_name"]].to_string(index=False))

rows = bat[bat["fielder_id"].isin(orphans)]
print(f"\ncatches involved: {len(rows)}")
print(rows[["match_id", "team", "player", "dismissal_kind", "fielder"]]
      .to_string(index=False))

orphan fielder ids: 17

how many are in the registry: 17 of 17
cricsheet_id     player_name
    04985f66        MJ Horne
    10d94c90       Abdul Haq
    149d0c18 PF Younghusband
    235a247a       RG Nijman
    2817e17a        EJ Black
    2c25d4f5      D Padikkal
    2c66ee1e       OGT Gould
    43501c0d         AW Gale
    47dc9794       MW Harvey
    8160713d  AW Hawkins-Kay
    95d3c4a6        TS Braat
    af28816c         TE Kane
    b67bfaae     GE Mathurin
    be869ccf      GHS Garton
    d684b87e      P de Bruyn
    d7ff1adc        CR Knott
    f57d5dbb        M Strano

catches involved: 23
 match_id                     team         player dismissal_kind         fielder
  1168519                 Zimbabwe      KM Jarvis         caught         TE Kane
  1239535                Sri Lanka   KIC Asalanka         caught      GHS Garton
  1262756                Sri Lanka   KIC Asalanka         caught      D Padikkal
  1262756                Sri Lanka   PVD Chameera         caught     

In [78]:
mk = pd.read_parquet(PROCESSED / "matches_keyed.parquet")
dv = pd.read_parquet(PROCESSED / "dim_venue.parquet")

pairs = [
    ("W.A.C.A. Ground", "Western Australia Cricket Association Ground"),
    ("Niranjan Shah Stadium", "Saurashtra Cricket Association Stadium"),
    ("Queenstown Events Centre", "John Davies Oval"),
]

for a, b in pairs:
    print(f"\n{'='*70}\n{a}   vs   {b}")
    sub = mk[mk["venue_clean"].isin([a, b])][
        ["match_date", "venue_clean", "city_final", "team1", "team2"]]
    sub = sub.sort_values("match_date")
    print(sub.to_string(index=False))

    # One ground cannot host two matches on the same day. If the two names
    # never share a date, they are consistent with being one ground.
    both = sub.groupby("match_date")["venue_clean"].nunique()
    print(f"  -> dates where BOTH names appear: {int((both > 1).sum())}")
    print(f"  -> date ranges: "
          f"{a}: {sub[sub.venue_clean==a].match_date.min().date()} to "
          f"{sub[sub.venue_clean==a].match_date.max().date()}   |   "
          f"{b}: {sub[sub.venue_clean==b].match_date.min().date()} to "
          f"{sub[sub.venue_clean==b].match_date.max().date()}")

print(f"\n{'='*70}\nvenues with a suspicious city")
print(dv[dv["venue_name"].isin(
    ["Windsor Park", "Chilaw Marians Cricket Club Ground", "Niaz Stadium"])]
    [["venue_id", "venue_name", "city", "country", "matches_played"]]
    .to_string(index=False))


W.A.C.A. Ground   vs   Western Australia Cricket Association Ground
match_date                                  venue_clean city_final        team1                team2
2004-02-01                              W.A.C.A. Ground      Perth        India            Australia
2005-01-30                              W.A.C.A. Ground      Perth    Australia             Pakistan
2005-02-01                              W.A.C.A. Ground      Perth     Pakistan          West Indies
2006-01-29 Western Australia Cricket Association Ground      Perth    Australia            Sri Lanka
2006-01-31 Western Australia Cricket Association Ground      Perth South Africa            Sri Lanka
2007-01-28 Western Australia Cricket Association Ground      Perth    Australia          New Zealand
2007-01-30 Western Australia Cricket Association Ground      Perth      England          New Zealand
2008-02-15 Western Australia Cricket Association Ground      Perth    Australia            Sri Lanka
2009-01-30 Western Aus

In [79]:
dv = pd.read_parquet(PROCESSED / "dim_venue.parquet")

print("--- the three merges ---")
print(dv[dv["venue_name"].isin([
    "Western Australia Cricket Association Ground",
    "Niranjan Shah Stadium", "John Davies Oval"])]
    [["venue_name", "city", "capacity", "former_names", "matches_played"]]
    .to_string(index=False))

print("\n--- the three city fixes ---")
print(dv[dv["venue_name"].isin([
    "Windsor Park", "Niaz Stadium", "Chilaw Marians Cricket Club Ground"])]
    [["venue_name", "city", "country", "matches_played"]]
    .to_string(index=False))

hit = dv["capacity"].notna()
print(f"\ncapacity: {hit.sum()} of {len(dv)} venues, "
      f"{dv.loc[hit,'matches_played'].sum():,} of {dv['matches_played'].sum():,} matches "
      f"({dv.loc[hit,'matches_played'].sum()/dv['matches_played'].sum():.0%})")

--- the three merges ---
                                  venue_name       city  capacity                           former_names  matches_played
                            John Davies Oval Queenstown   19000.0  Davies Park, Queenstown Events Centre              13
                       Niranjan Shah Stadium     Rajkot   28000.0 Saurashtra Cricket Association Stadium               8
Western Australia Cricket Association Ground      Perth   20000.0                        W.A.C.A. Ground              28

--- the three city fixes ---
                        venue_name       city   country  matches_played
Chilaw Marians Cricket Club Ground Katunayake Sri Lanka               1
                      Niaz Stadium  Hyderabad  Pakistan               1
                      Windsor Park     Roseau  Dominica               4

capacity: 192 of 223 venues, 3,098 of 3,182 matches (97%)


In [80]:
gap = dv[dv["capacity"].isna()].sort_values("matches_played", ascending=False)
print(f"{len(gap)} venues with no capacity, {gap['matches_played'].sum()} matches\n")
print(gap[["venue_name", "city", "country", "matches_played"]]
      .head(15).to_string(index=False))

31 venues with no capacity, 84 matches

                                venue_name          city      country  matches_played
                  Wanderers Cricket Ground      Windhoek      Namibia              21
                    Takashinga Sports Club        Harare     Zimbabwe               9
    Lahore City Cricket Association Ground        Lahore     Pakistan               7
                             Old Hararians        Harare     Zimbabwe               5
                 Royal Chiangmai Golf Club    Chiang Mai     Thailand               4
                              Windsor Park        Roseau     Dominica               4
          North-West University No1 Ground Potchefstroom South Africa               3
                     Cambusdoon New Ground           Ayr     Scotland               3
Sheikh Kamal International Cricket Stadium   Cox's Bazar   Bangladesh               2
                 Stellenbosch University 1  Stellenbosch South Africa               2
              

In [81]:
dv = pd.read_parquet(PROCESSED / "dim_venue.parquet")
hit = dv["capacity"].notna()
print(f"venues with capacity: {hit.sum()} of {len(dv)}")
print(f"matches covered     : {dv.loc[hit,'matches_played'].sum():,} of "
      f"{dv['matches_played'].sum():,} "
      f"({dv.loc[hit,'matches_played'].sum()/dv['matches_played'].sum():.0%})")
print()
print(dv[dv["venue_name"] == "Windsor Park"]
      [["venue_name", "city", "country", "capacity", "capacity_source"]]
      .to_string(index=False))

venues with capacity: 194 of 223
matches covered     : 3,103 of 3,182 (98%)

  venue_name   city  country  capacity capacity_source
Windsor Park Roseau Dominica   12000.0  wikipedia-name


In [82]:
mk = pd.read_parquet(PROCESSED / "matches_keyed.parquet")
dt = pd.read_parquet(PROCESSED / "dim_team.parquet")

# Stated by Cricsheet on their downloads page, 14 Nov 2024.
WITHHELD_ODIS = 159

held = len(mk)
in_scope = held + WITHHELD_ODIS

print(f"ODIs in our archive    : {held:,}")
print(f"withheld by the source : {WITHHELD_ODIS}")
print(f"ODIs actually played   : {in_scope:,}")
print(f"coverage               : {held / in_scope:.1%}")
print(f"\nAfghanistan present in dim_team: "
      f"{'Afghanistan' in set(dt['team_name'])}")
print(f"teams we do have: {dt['team_name'].nunique()}")

print(f"\nour window: {mk['match_date'].min().date()} to "
      f"{mk['match_date'].max().date()}")
print("Afghanistan's first ODI: 19 Apr 2009 - entirely inside our window,")
print("so this is a clean exclusion, not a date-range artefact.")

ODIs in our archive    : 3,182
withheld by the source : 159
ODIs actually played   : 3,341
coverage               : 95.2%

Afghanistan present in dim_team: False
teams we do have: 28

our window: 2002-06-27 to 2026-09-09
Afghanistan's first ODI: 19 Apr 2009 - entirely inside our window,
so this is a clean exclusion, not a date-range artefact.


In [83]:
bat = pd.read_parquet(PROCESSED / "batting.parquet")
bwl = pd.read_parquet(PROCESSED / "bowling.parquet")

for name, df in [("batting", bat), ("bowling", bwl)]:
    print(f"\n{'='*66}\n{name}.parquet — {len(df):,} rows\n{'='*66}")
    summary = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "nulls": df.isna().sum(),
        "null_%": (df.isna().sum() / len(df) * 100).round(1),
        "distinct": df.nunique(),
    })
    # min/max only where it makes sense
    nums = df.select_dtypes("number")
    summary["min"] = nums.min()
    summary["max"] = nums.max()
    print(summary.to_string())


batting.parquet — 56,052 rows
                  dtype  nulls  null_%  distinct      min        max
player              str      0     0.0      2582      NaN        NaN
team                str      0     0.0        28      NaN        NaN
innings_no        int64      0     0.0         2      1.0        2.0
batting_position  int64      0     0.0        12      1.0       12.0
runs_scored       int64      0     0.0       198      0.0      264.0
balls_faced       int64      0     0.0       169      0.0      173.0
fours             int64      0     0.0        29      0.0       33.0
sixes             int64      0     0.0        17      0.0       16.0
dismissal_kind      str  10003    17.8         9      NaN        NaN
dismissed_by        str  13501    24.1      1691      NaN        NaN
fielder             str  26320    47.0      2135      NaN        NaN
is_not_out         bool      0     0.0         2      NaN        NaN
match_id          int64      0     0.0      3182  64814.0  1549973.0
pla

In [84]:
mk = pd.read_parquet(PROCESSED / "matches_keyed.parquet")

odd = bwl[bwl["balls_bowled"] > 60]
print(f"spells longer than 10 overs: {len(odd)}\n")
print(odd.sort_values("balls_bowled", ascending=False)
      [["match_id", "player", "team", "innings_no",
        "balls_bowled", "runs_conceded", "wickets"]]
      .head(15).to_string(index=False))

print("\n--- what kind of matches were those? ---")
cols = [c for c in ["match_id", "match_date", "gender", "team1", "team2",
                    "scheduled_overs", "balls_per_over", "series_name"]
        if c in mk.columns]
print(mk[mk["match_id"].isin(odd["match_id"].unique())][cols]
      .drop_duplicates().to_string(index=False))

zero = bwl[bwl["balls_bowled"] == 0]
print(f"\n--- bowlers with zero legal balls: {len(zero)} ---")
print(zero[["match_id", "player", "team", "balls_bowled",
            "runs_conceded", "wickets"]].head(10).to_string(index=False))

spells longer than 10 overs: 17

 match_id        player         team  innings_no  balls_bowled  runs_conceded  wickets
  1379758     EJ Carson  New Zealand           2            66             41        2
  1022373      R Ashwin        India           1            61             54        0
  1153693     HH Pandya        India           1            61             45        2
  1322031      DS Airee        Nepal           1            61             41        1
  1243925   RS Gayakwad        India           2            61             48        0
  1325551 Kuldeep Yadav        India           2            61             38        2
  1339596      M Jansen South Africa           1            61             66        1
  1395701  Mahedi Hasan   Bangladesh           1            61             45        3
  1477010      K Fraser     Scotland           2            61             50        3
   258475     SCJ Broad      England           1            61             84        1
   289110 

In [85]:
m = 1379758

cols = [c for c in ["match_id", "match_date", "gender", "team1", "team2",
                    "scheduled_overs", "balls_per_over", "winner",
                    "victory_type", "victory_margin", "series_name"]
        if c in mk.columns]
print(mk[mk["match_id"] == m][cols].to_string(index=False))

spell = bwl[bwl["match_id"] == m][
    ["player", "team", "innings_no", "balls_bowled",
     "runs_conceded", "wickets", "maidens", "dots"]]
print("\n--- bowling ---")
print(spell.sort_values(["innings_no", "balls_bowled"],
                        ascending=[True, False]).to_string(index=False))

print("\n--- legal balls per innings (300 = a full 50 overs) ---")
print(spell.groupby("innings_no")["balls_bowled"].agg(["sum", "count"])
      .rename(columns={"sum": "legal_balls", "count": "bowlers"}).to_string())

print("\n--- batting, innings 1 totals ---")
b = bat[bat["match_id"] == m]
print(b.groupby("innings_no")[["runs_scored", "balls_faced"]].sum().to_string())

 match_id match_date gender       team1     team2  scheduled_overs  balls_per_over      winner victory_type  victory_margin              series_name
  1379758 2023-06-30 female New Zealand Sri Lanka               50               6 New Zealand         runs           116.0 ICC Women's Championship

--- bowling ---
        player        team  innings_no  balls_bowled  runs_conceded  wickets  maidens  dots
 OU Ranasinghe   Sri Lanka           1            60             68        3        0    15
   BMSM Kumari   Sri Lanka           1            60             79        1        0    19
KDU Prabodhani   Sri Lanka           1            54             38        2        0    31
   I Ranaweera   Sri Lanka           1            48             56        1        0    17
    WK Dilhari   Sri Lanka           1            42             44        0        0    16
  AC Jayangani   Sri Lanka           1            24             23        0        0     7
     K Kavindi   Sri Lanka           1   

In [86]:
import zipfile, json
from collections import Counter

zpath = root / "data" / "raw" / "odis_json.zip"
with zipfile.ZipFile(zpath) as z:
    data = json.load(z.open("1379758.json"))

inn = data["innings"][1]
print(f"batting team: {inn['team']}   overs recorded: {len(inn['overs'])}")

legal, total = Counter(), Counter()
for over in inn["overs"]:
    for d in over["deliveries"]:
        b = d["bowler"]
        total[b] += 1
        ex = d.get("extras", {})
        if "wides" not in ex and "noballs" not in ex:
            legal[b] += 1

print("\nlegal balls per bowler, straight from the JSON:")
for b, n in legal.most_common():
    print(f"  {b:22s} legal {n:3d}  deliveries {total[b]:3d}  = {n//6}.{n%6} ov")

print("\nCarson, over by over:")
nums = []
for over in inn["overs"]:
    ds = [d for d in over["deliveries"] if d["bowler"] == "EJ Carson"]
    if ds:
        lg = sum(1 for d in ds
                 if not ({"wides", "noballs"} & set(d.get("extras", {}))))
        nums.append(over["over"])
        print(f"  over {over['over']:2d}: {len(ds)} deliveries, {lg} legal")

print(f"\ndistinct over numbers she bowled: {len(nums)}")
print(f"any over number repeated: {len(nums) != len(set(nums))}")

batting team: Sri Lanka   overs recorded: 49

legal balls per bowler, straight from the JSON:
  EJ Carson              legal  66  deliveries  69  = 11.0 ov
  AC Kerr                legal  60  deliveries  63  = 10.0 ov
  LMM Tahuhu             legal  48  deliveries  53  = 8.0 ov
  HM Rowe                legal  42  deliveries  47  = 7.0 ov
  FC Jonas               legal  34  deliveries  35  = 5.4 ov
  SFM Devine             legal  24  deliveries  27  = 4.0 ov
  BM Halliday            legal  18  deliveries  20  = 3.0 ov

Carson, over by over:
  over  6: 6 deliveries, 6 legal
  over  8: 6 deliveries, 6 legal
  over 16: 7 deliveries, 6 legal
  over 18: 6 deliveries, 6 legal
  over 20: 6 deliveries, 6 legal
  over 22: 7 deliveries, 6 legal
  over 30: 6 deliveries, 6 legal
  over 32: 7 deliveries, 6 legal
  over 42: 6 deliveries, 6 legal
  over 44: 6 deliveries, 6 legal
  over 46: 6 deliveries, 6 legal

distinct over numbers she bowled: 11
any over number repeated: False


In [87]:
bat = pd.read_parquet(PROCESSED / "batting.parquet")
bwl = pd.read_parquet(PROCESSED / "bowling.parquet")
mk = pd.read_parquet(PROCESSED / "matches_keyed.parquet")

# A primary key you haven't proved unique is a guess.
for name, df in [("batting", bat), ("bowling", bwl)]:
    k = ["match_id", "innings_no", "player_id"]
    dupes = int(df.duplicated(k).sum())
    print(f"{name}: duplicate (match_id, innings_no, player_id) = {dupes}")
    if dupes:
        d = df[df.duplicated(k, keep=False)].sort_values(k)
        print(d.head(12).to_string(index=False))

# player_of_match never got resolved to an id.
print(f"\nplayer_of_match dtype : {mk['player_of_match'].dtype}")
print(f"nulls                 : {int(mk['player_of_match'].isna().sum())}")
print(f"distinct              : {mk['player_of_match'].nunique()}")
print("sample:", mk["player_of_match"].dropna().head(5).tolist())

batting: duplicate (match_id, innings_no, player_id) = 0
bowling: duplicate (match_id, innings_no, player_id) = 0

player_of_match dtype : str
nulls                 : 222
distinct              : 897
sample: ['MS Wade', 'Mohammad Hafeez', 'SPD Smith', 'DA Warner', 'DA Warner']


In [88]:
squads = pd.read_parquet(PROCESSED / "squads.parquet")

# Could one name appear twice in a single match's team sheets?
# If so, per-match resolution is ambiguous too and we'd need another way.
dupe = int(squads.duplicated(["match_id", "player_name"]).sum())
print(f"duplicate (match_id, player_name) in squads: {dupe}")

lookup = (squads.drop_duplicates(["match_id", "player_name"])
          .set_index(["match_id", "player_name"])["player_id"])

mk["player_of_match_id"] = pd.MultiIndex.from_arrays(
    [mk["match_id"], mk["player_of_match"]]).map(lookup)

named = mk["player_of_match"].notna()
resolved = mk["player_of_match_id"].notna()

print(f"\nmatches with an award : {int(named.sum()):,}")
print(f"resolved to a player  : {int(resolved.sum()):,}")
print(f"unresolved            : {int((named & ~resolved).sum())}")

bad = mk[named & ~resolved]
if len(bad):
    print("\n--- names that did not resolve ---")
    print(bad[["match_id", "match_date", "team1", "team2", "player_of_match"]]
          .head(20).to_string(index=False))

print(f"\nmatches with no award : {int((~named).sum())}")
print("of which had no winner:",
      int(mk.loc[~named, "winner"].isna().sum()))

duplicate (match_id, player_name) in squads: 0

matches with an award : 2,960
resolved to a player  : 2,960
unresolved            : 0

matches with no award : 222
of which had no winner: 120


In [89]:
b = bat.groupby(["match_id", "innings_no"])["runs_scored"].sum().rename("off_the_bat")
w = bwl.groupby(["match_id", "innings_no"])["runs_conceded"].sum().rename("charged_to_bowlers")
cmp = pd.concat([b, w], axis=1).reset_index()
cmp["gap"] = cmp["charged_to_bowlers"] - cmp["off_the_bat"]

print(f"innings: {len(cmp):,}")
print(f"innings where the two agree: {(cmp['gap'] == 0).sum():,}")
print(f"\ngap (wides + no-balls only; byes and leg-byes are in neither):")
print(cmp["gap"].describe().to_string())

print("\n--- the highest 'totals' we could currently report ---")
top = cmp.sort_values("off_the_bat", ascending=False).head(5)
top = top.merge(mk[["match_id", "match_date", "team1", "team2"]], on="match_id")
print(top.to_string(index=False))
print("\nThe real scores for these matches are higher than 'off_the_bat',")
print("by the extras we cannot see.")

innings: 6,283
innings where the two agree: 63

gap (wides + no-balls only; byes and leg-byes are in neither):
count    6283.000000
mean        8.801528
std         5.291164
min         0.000000
25%         5.000000
50%         8.000000
75%        12.000000
max        43.000000

--- the highest 'totals' we could currently report ---
 match_id  innings_no  off_the_bat  charged_to_bowlers  gap match_date        team1        team2
  1281444           1          476                 492   16 2022-06-17      England  Netherlands
  1119539           1          462                 472   10 2018-06-19      England    Australia
   722341           1          430                 435    5 2015-01-18 South Africa  West Indies
   903601           1          427                 436    9 2015-10-25        India South Africa
   238200           2          418                 426    8 2006-03-12 South Africa    Australia

The real scores for these matches are higher than 'off_the_bat',
by the extras we 